# moderate ordinal model v6 Reviewer-Closure Notebook

Bu notebook, v5 ana 3-seed sonuçları zaten üretildikten sonra kalan hakem açıklarını kapatmak için hazırlanmıştır.

Ana amaç:

1. Kaggle input olarak eklenen önceki v5 çıktılarını içeri almak.
2. Ana 3-seed CE vs moderate ordinal model sonuçlarını tekrar eğitmeden korumak.
3. Eksik kalan reviewer tablolarını üretmek:
   - SE / no-SE ablation
   - CE / weighted CE / focal / CORAL / pairwise ablation
   - λ / δ sensitivity
   - complexity table
   - bootstrap CI güncellemesi
   - anonim repo paketi

Önceki çıktı dataset path’i varsayılan olarak şudur:

```text
/kaggle/input/<anonymous_previous_outputs>/previous-main-outputs
```

Bu path yalnızca çalıştırma sırasında önceki CSV/JSON/NPY çıktıları içeri almak için kullanılır. Anonim repo paketine bu kişisel Kaggle path’i yazılmaz.


In [ ]:
# =========================
# 0) GLOBAL CONFIG — v6 REVIEWER-CLOSURE DEFAULTS
# =========================

from pathlib import Path

CONFIG = {
    # Dataset root must contain train / val(or valid/validation) / test folders.
    # Each split must contain grade folders such as 0,1,2,3,4 or names beginning with 0..4.
    "DATA_ROOT": "",

    # Output folder
    "OUT_DIR": "./reviewer_closure_outputs",

    # Main run settings
    # EfficientNet-B4 @ 380 px gave the first useful signal, but it is NOT a lightweight model.
    # Report it as a stronger single-model revision, not as "lightweight".
    "IMG_SIZE": 380,
    "BATCH_SIZE": 8,
    "NUM_WORKERS": 2,
    "EPOCHS": 42,
    "PATIENCE": 10,
    "SEEDS": [42, 7, 123],

    # Options implemented: efficientnet_b0, efficientnet_b3, efficientnet_b4,
    # efficientnet_v2_s, convnext_tiny, resnet50
    "BACKBONE": "efficientnet_b4",
    "PRETRAINED": True,
    "USE_EXTRA_SE": True,
    "DROPOUT": 0.30,

    # ROI / preprocessing
    "USE_ROI_AUTOCROP": True,

    # Loss settings.
    # These defaults are overridden by MAIN_EXPERIMENTS for each arm.
    # They remain nonzero to prevent accidental CE-only "moderate ordinal model" runs.
    "LOSS_MODE": "weighted_ce",  # "ce", "weighted_ce", "focal", "cb_focal"
    "LABEL_SMOOTHING": 0.02,
    "FOCAL_GAMMA": 1.0,
    "CB_BETA": 0.999,
    "CE_WEIGHT": 1.00,
    "ORDINAL_WEIGHT": 0.15,
    "RANK_WEIGHT": 0.08,
    "PAIRWISE_MARGIN": 1.0,
    "ORDINAL_PROB_BLEND": 0.10,

    # Sampling / imbalance
    "USE_WEIGHTED_SAMPLER": True,
    "SAMPLER_POWER": 0.25,       # mild balancing; avoids Grade-1 collapse without destroying exact accuracy
    "GRADE1_EXTRA_WEIGHT": 1.00, # do not over-push Grade 1 during the main accuracy run

    # Model selection — exact accuracy first, QWK tie-breaker.
    # Grade-1 recall is reported, not used as an uncontrolled selection target.
    "VAL_SCORE_ACC_W": 0.85,
    "VAL_SCORE_QWK_W": 0.15,
    "VAL_SCORE_G1_W": 0.00,

    # Test-time evaluation
    "USE_TTA_HFLIP": True,
    "RUN_CLASS_BIAS_TUNING": False,

    # Main and reviewer experiment switches
    "RUN_MAIN_EXPERIMENTS": False,
    "RUN_FULL_ABLATION": True,
    "RUN_LAMBDA_DELTA_SENSITIVITY": True,
    "RUN_MODEL_ZOO_SWEEP": False,

    # Reviewer extras are one-seed by default because B4/380 is expensive.
    # Main CE/OA results are imported from the previous 3-seed run.
    # For maximum-strength reviewer tables, set to [42, 7, 123] and keep EPOCHS=42.
    "REVIEWER_TABLE_SEEDS": [42],
    "REVIEWER_TABLE_EPOCHS": 42,
    "REVIEWER_TABLE_PATIENCE": 10,

    # Optional Grade-0/1 boundary refiner. Keep off unless main model is strong and you explicitly report it as post-hoc.
    "RUN_GRADE01_REFINER": False,
    "REFINER_EPOCHS": 20,
    "REFINER_BACKBONE": "efficientnet_b0",

    # Model-zoo sweep: use only if B4 also stalls.
    "SWEEP_EPOCHS": 18,
    "SWEEP_PATIENCE": 6,
    "SWEEP_SEED": 42,
    "MODEL_ZOO_SWEEP": [
        {"name": "b3_300", "BACKBONE": "efficientnet_b3", "IMG_SIZE": 300, "BATCH_SIZE": 16, "DROPOUT": 0.30},
        {"name": "b4_380", "BACKBONE": "efficientnet_b4", "IMG_SIZE": 380, "BATCH_SIZE": 8, "DROPOUT": 0.30},
        {"name": "v2s_384", "BACKBONE": "efficientnet_v2_s", "IMG_SIZE": 384, "BATCH_SIZE": 8, "DROPOUT": 0.25},
        {"name": "convnext_tiny_384", "BACKBONE": "convnext_tiny", "IMG_SIZE": 384, "BATCH_SIZE": 8, "DROPOUT": 0.25},
    ],

    # Training
    "LR_BACKBONE": 1.5e-5,
    "LR_HEAD": 1.5e-4,
    "WEIGHT_DECAY": 1e-4,
    "GRAD_CLIP": 1.0,
    "USE_AMP": True,

    # Repo/reproducibility output settings
    "SAVE_CHECKPOINTS": False,
    "MAKE_ANON_REPO_PACKAGE": True,
    "REPO_PACKAGE_NAME": "ordinal_model_v6_anonymous_repo_package",

    # Previous v5 output package imported as a Kaggle dataset.
    # Used to avoid rerunning the finished 3-seed main CE/OA experiments.
    "IMPORT_PREVIOUS_OUTPUTS": True,
    "PREVIOUS_OUTPUTS_ROOT": "/kaggle/input/<anonymous_previous_outputs>/previous-main-outputs",
    # Optional: add the failed/partial v6 output dataset path here after saving it as a Kaggle dataset.
    # Example: ["/kaggle/input/<anonymous_previous_outputs>/<v6-partial-output-dataset>"]
    "PREVIOUS_OUTPUTS_EXTRA_ROOTS": [],
    "USE_IMPORTED_MAIN_FOR_REVIEWER_TABLES": True,
    "RESUME_COMPLETED_REVIEWER_RUNS": True,
    "ANONYMIZE_IMPORTED_PATHS_IN_REPO": True,
}

OUT_DIR = Path(CONFIG["OUT_DIR"])
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Output directory:", OUT_DIR.resolve())

# =========================
# 0B) MAIN EXPERIMENT ARMS — same backbone, same image size, same seeds
# =========================
# This is the core evidence for the paper.
#
# ce_baseline_*:
#   Strong classifier baseline with the same backbone/resolution/SE/sampler.
#
# moderate_ordinal_*:
#   Same architecture and schedule, but CORAL + pairwise ranking are active.
#   The weights are deliberately moderate: strong enough to be methodologically real,
#   not so high that exact accuracy collapses. Sensitivity rows below test stronger values.
#
# Report both arms. If moderate ordinal model does not beat CE on QWK/MAE under this fair setup,
# do not claim that the ordinal loss improves performance.

MAIN_EXPERIMENTS = [
    {
        "name": f"ce_baseline_{CONFIG['BACKBONE']}_{CONFIG['IMG_SIZE']}",
        "LOSS_MODE": "weighted_ce",
        "USE_EXTRA_SE": True,
        "ORDINAL_WEIGHT": 0.0,
        "RANK_WEIGHT": 0.0,
        "ORDINAL_PROB_BLEND": 0.0,
        "USE_WEIGHTED_SAMPLER": True,
        "SAMPLER_POWER": 0.25,
        "GRADE1_EXTRA_WEIGHT": 1.00,
    },
    {
        "name": f"moderate_ordinal_{CONFIG['BACKBONE']}_{CONFIG['IMG_SIZE']}",
        "LOSS_MODE": "weighted_ce",
        "USE_EXTRA_SE": True,
        "ORDINAL_WEIGHT": 0.15,
        "RANK_WEIGHT": 0.08,
        "ORDINAL_PROB_BLEND": 0.10,
        "PAIRWISE_MARGIN": 1.0,
        "USE_WEIGHTED_SAMPLER": True,
        "SAMPLER_POWER": 0.25,
        "GRADE1_EXTRA_WEIGHT": 1.00,
    },
]

print("Main experiment arms:")
for arm in MAIN_EXPERIMENTS:
    print("  -", arm["name"], "| ORDINAL_WEIGHT=", arm["ORDINAL_WEIGHT"], "| RANK_WEIGHT=", arm["RANK_WEIGHT"])


In [ ]:
# =========================
# 0C) IMPORT PREVIOUS V5 MAIN OUTPUTS FROM KAGGLE INPUT
# =========================
# The v5 main CE vs moderate ordinal model 3-seed run is already complete.
# This cell copies those outputs from the Kaggle input dataset into the current OUT_DIR,
# so the final repo package contains both the old main results and the new reviewer-closure tables.
#
# It supports either:
#   1) a folder named main_experiment_outputs, or
#   2) a zip file containing that folder.

import shutil
import zipfile
import json
from pathlib import Path

def _looks_like_oa_output_dir(p: Path) -> bool:
    if not p.is_dir():
        return False
    indicators = [
        p / "ALL_SUMMARIES_COLLECTED.csv",
        p / "bootstrap_ci_all_ensemble_outputs.csv",
        p / "significance_tests",
        p / "ce_baseline_efficientnet_b4_380",
        p / "moderate_ordinal_efficientnet_b4_380",
    ]
    return any(x.exists() for x in indicators)

def _copy_output_tree(src: Path, dst: Path):
    copied = 0
    skipped = 0
    dst.mkdir(parents=True, exist_ok=True)

    for item in src.rglob("*"):
        if not item.is_file():
            continue

        rel = item.relative_to(src)
        rel_parts_lower = [part.lower() for part in rel.parts]
        name_lower = item.name.lower()

        # Avoid carrying previous anonymous repo package recursively.
        if any("anonymous_repo_package" in part for part in rel_parts_lower):
            skipped += 1
            continue

        # Keep result artifacts, but avoid accidental manuscript/submission files.
        blocked_suffixes = {".doc", ".docx", ".tex", ".bib", ".pdf", ".pt", ".pth", ".ckpt", ".onnx"}
        blocked_tokens = ["manuscript", "marked", "clear", "cover", "response", "reviewer", "editor", "submission"]
        if item.suffix.lower() in blocked_suffixes or any(tok in name_lower for tok in blocked_tokens):
            skipped += 1
            continue

        out_file = dst / rel
        out_file.parent.mkdir(parents=True, exist_ok=True)
        if out_file.exists():
            # Do not overwrite newly generated reviewer-closure outputs.
            skipped += 1
            continue
        shutil.copy2(item, out_file)
        copied += 1

    return copied, skipped

def find_previous_outputs_root(root: Path, extract_root: Path):
    root = Path(root)
    if not root.exists():
        return None, f"Previous output root does not exist: {root}"

    # Direct folder candidates.
    direct_candidates = []
    if _looks_like_oa_output_dir(root):
        direct_candidates.append(root)
    direct_candidates += [p for p in root.rglob("*") if _looks_like_oa_output_dir(p)]

    if direct_candidates:
        direct_candidates = sorted(direct_candidates, key=lambda p: len(p.parts))
        return direct_candidates[0], "folder"

    # Zip candidates.
    zip_files = sorted(root.rglob("*.zip"))
    for zpath in zip_files:
        try:
            z_extract = extract_root / zpath.stem.replace(" ", "_")
            if not z_extract.exists():
                z_extract.mkdir(parents=True, exist_ok=True)
                with zipfile.ZipFile(zpath, "r") as zf:
                    zf.extractall(z_extract)
            extracted_candidates = [p for p in z_extract.rglob("*") if _looks_like_oa_output_dir(p)]
            if _looks_like_oa_output_dir(z_extract):
                extracted_candidates.append(z_extract)
            if extracted_candidates:
                extracted_candidates = sorted(extracted_candidates, key=lambda p: len(p.parts))
                return extracted_candidates[0], f"zip:{zpath.name}"
        except Exception as e:
            print(f"Could not inspect/extract {zpath}: {repr(e)}")

    return None, "No previous moderate ordinal model output folder or zip was found."

if CONFIG.get("IMPORT_PREVIOUS_OUTPUTS", False):
    roots_to_try = []
    primary = CONFIG.get("PREVIOUS_OUTPUTS_ROOT", "")
    if primary:
        roots_to_try.append(primary)
    roots_to_try.extend(CONFIG.get("PREVIOUS_OUTPUTS_EXTRA_ROOTS", []) or [])

    imported_manifests = []
    total_copied, total_skipped = 0, 0

    for root_i, root_item in enumerate(roots_to_try, start=1):
        previous_root = Path(root_item)
        imported_extract_root = OUT_DIR / "_imported_previous_zip_extract" / f"root_{root_i}"
        previous_output_dir, previous_source_kind = find_previous_outputs_root(previous_root, imported_extract_root)

        if previous_output_dir is None:
            print("WARNING:", previous_source_kind)
            continue

        copied, skipped = _copy_output_tree(previous_output_dir, OUT_DIR)
        total_copied += copied
        total_skipped += skipped
        imported_manifests.append({
            "source_kind": previous_source_kind,
            "source_path_redacted": f"<KAGGLE_INPUT_PREVIOUS_OUTPUTS_{root_i}>",
            "copied_files": int(copied),
            "skipped_files": int(skipped),
        })
        print(f"Imported previous outputs root {root_i} into {OUT_DIR.resolve()}")
        print(f"  copied files: {copied} | skipped files: {skipped}")

    if not imported_manifests:
        print("WARNING: No previous output folders/zips were imported.")
        print("Ablation/sensitivity will still run, but previous main/partial outputs may be missing.")
    else:
        import_manifest = {
            "imported_previous_outputs": True,
            "num_roots_imported": len(imported_manifests),
            "roots": imported_manifests,
            "total_copied_files": int(total_copied),
            "total_skipped_files": int(total_skipped),
            "note": "Previous outputs were copied into the v6 OUT_DIR. Personal Kaggle paths are redacted for anonymous packaging."
        }
        OUT_DIR.mkdir(parents=True, exist_ok=True)
        (OUT_DIR / "previous_output_import_manifest.json").write_text(json.dumps(import_manifest, indent=2), encoding="utf-8")
else:
    print("CONFIG['IMPORT_PREVIOUS_OUTPUTS'] is False; previous outputs were not imported.")


In [ ]:
# =========================
# 1) IMPORTS AND ENV
# =========================

import os, re, time, math, random, json, copy, warnings
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import pandas as pd

from PIL import Image, ImageOps

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, Subset
from torchvision import transforms, models

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, cohen_kappa_score,
    mean_absolute_error, precision_recall_fscore_support,
    confusion_matrix, classification_report
)

warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("Torch:", torch.__version__)

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # For reproducibility. Turning benchmark off can slow training.
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(42)

In [ ]:
# =========================
# 2) DATASET DISCOVERY
# =========================

def find_data_root(user_root: str = ""):
    candidates = []
    if user_root:
        candidates.append(Path(user_root))

    # Common Kaggle / Colab / local candidates
    candidates += [
        Path("/kaggle/input/<anonymous_previous_outputs>/knee-osteoarthritis-dataset-with-severity"),
        Path("/kaggle/input/knee-osteoarthritis-dataset-with-severity/kneeOA"),
        Path("/kaggle/input/knee-osteoarthritis-dataset-with-severity/Knee Osteoarthritis Dataset with Severity"),
        Path("/kaggle/input"),
        Path("/content"),
        Path("/mnt/data"),
        Path("."),
    ]

    split_names = [
        ("train", "val", "test"),
        ("train", "valid", "test"),
        ("train", "validation", "test"),
        ("Training", "Validation", "Testing"),
    ]

    def has_splits(p: Path):
        if not p.exists() or not p.is_dir():
            return None
        children = {c.name.lower(): c for c in p.iterdir() if c.is_dir()}
        for tr, va, te in split_names:
            if tr.lower() in children and va.lower() in children and te.lower() in children:
                return {
                    "train": children[tr.lower()],
                    "val": children[va.lower()],
                    "test": children[te.lower()],
                }
        return None

    # Direct candidates
    for c in candidates:
        r = has_splits(c)
        if r:
            return c, r

    # Search two levels deep in Kaggle input
    for base in [Path("/kaggle/input"), Path("/content"), Path(".")]:
        if base.exists():
            for p in list(base.glob("*")) + list(base.glob("*/*")):
                r = has_splits(p)
                if r:
                    return p, r

    raise FileNotFoundError(
        "Dataset root not found. Set CONFIG['DATA_ROOT'] to a folder containing train/val/test."
    )

DATA_ROOT, SPLITS = find_data_root(CONFIG["DATA_ROOT"])
print("Detected DATA_ROOT:", DATA_ROOT)
print("Splits:")
for k,v in SPLITS.items():
    print(f"  {k}: {v}")

In [ ]:
# =========================
# 3) DATASET CLASS
# =========================

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

def parse_grade_from_folder(folder_name: str):
    # Works for: "0", "1", "Grade 2", "2Moderate", "3_severe", etc.
    m = re.search(r"([0-4])", folder_name)
    if m:
        return int(m.group(1))
    return None

class KneeOADataset(Dataset):
    def __init__(self, split_dir, transform=None, allowed_labels=None):
        self.split_dir = Path(split_dir)
        self.transform = transform
        self.samples = []
        self.allowed_labels = set(allowed_labels) if allowed_labels is not None else None

        for class_dir in sorted([p for p in self.split_dir.iterdir() if p.is_dir()]):
            y = parse_grade_from_folder(class_dir.name)
            if y is None:
                continue
            if self.allowed_labels is not None and y not in self.allowed_labels:
                continue
            for img_path in class_dir.rglob("*"):
                if img_path.suffix.lower() in IMG_EXTS:
                    self.samples.append((img_path, y))

        if not self.samples:
            raise RuntimeError(f"No image samples found in {self.split_dir}")

        self.samples = sorted(self.samples, key=lambda x: str(x[0]))
        self.labels = np.array([y for _, y in self.samples], dtype=np.int64)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, y = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(y, dtype=torch.long), torch.tensor(idx, dtype=torch.long)

def summarize_split(name, ds):
    cnt = Counter(ds.labels.tolist())
    row = {"split": name, "n": len(ds)}
    for g in range(5):
        row[f"grade_{g}"] = cnt.get(g, 0)
    return row

# Temporary no-transform datasets for counts
_tmp_train = KneeOADataset(SPLITS["train"], transform=None)
_tmp_val = KneeOADataset(SPLITS["val"], transform=None)
_tmp_test = KneeOADataset(SPLITS["test"], transform=None)

split_df = pd.DataFrame([
    summarize_split("train", _tmp_train),
    summarize_split("val", _tmp_val),
    summarize_split("test", _tmp_test),
])
display(split_df)
split_df.to_csv(OUT_DIR / "split_distribution.csv", index=False)

In [ ]:
# =========================
# 4) TRANSFORMS AND LOADERS
# =========================

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

class AutoContrast:
    def __call__(self, img):
        return ImageOps.autocontrast(img)

class CropNonEmpty:
    """Crop dark/empty borders in X-ray images before resizing.

    This is deliberately conservative. If no reliable mask is found, the image is returned unchanged.
    """
    def __init__(self, threshold=8, pad_ratio=0.03):
        self.threshold = threshold
        self.pad_ratio = pad_ratio

    def __call__(self, img):
        arr = np.asarray(img.convert("L"))
        mask = arr > self.threshold
        if mask.sum() < 50:
            return img
        ys, xs = np.where(mask)
        x0, x1 = xs.min(), xs.max() + 1
        y0, y1 = ys.min(), ys.max() + 1
        h, w = arr.shape
        pad = int(max(h, w) * self.pad_ratio)
        x0 = max(0, x0 - pad); y0 = max(0, y0 - pad)
        x1 = min(w, x1 + pad); y1 = min(h, y1 + pad)
        if (x1 - x0) < 0.40 * w or (y1 - y0) < 0.40 * h:
            return img
        return img.crop((x0, y0, x1, y1))

def build_transforms(img_size=224, use_roi_autocrop=True):
    pre = []
    if use_roi_autocrop:
        pre.append(CropNonEmpty(threshold=8, pad_ratio=0.03))
    pre.append(AutoContrast())

    # v3: high-resolution, conservative augmentation.
    # KL grading depends on joint-space details; avoid aggressive crop/rotation.
    train_tf = transforms.Compose(pre + [
        transforms.Resize(int(img_size * 1.08)),
        transforms.RandomResizedCrop(img_size, scale=(0.94, 1.00), ratio=(0.97, 1.03)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=3),
        transforms.ColorJitter(brightness=0.04, contrast=0.08),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

    eval_tf = transforms.Compose(pre + [
        transforms.Resize(int(img_size * 1.08)),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    return train_tf, eval_tf

def class_counts_from_dataset(ds, num_classes=5):
    cnt = np.bincount(ds.labels, minlength=num_classes)
    return cnt

def make_sampler(ds, sampler_power=0.65, grade1_extra=1.35):
    counts = class_counts_from_dataset(ds)
    inv = 1.0 / np.maximum(counts, 1)
    weights_per_class = inv ** sampler_power
    weights_per_class[1] *= grade1_extra
    sample_weights = weights_per_class[ds.labels]
    return WeightedRandomSampler(
        weights=torch.DoubleTensor(sample_weights),
        num_samples=len(sample_weights),
        replacement=True,
    )

def build_loaders(config, allowed_labels=None, shuffle_train=True):
    train_tf, eval_tf = build_transforms(
        config["IMG_SIZE"],
        use_roi_autocrop=config.get("USE_ROI_AUTOCROP", True),
    )
    train_ds = KneeOADataset(SPLITS["train"], transform=train_tf, allowed_labels=allowed_labels)
    val_ds = KneeOADataset(SPLITS["val"], transform=eval_tf, allowed_labels=allowed_labels)
    test_ds = KneeOADataset(SPLITS["test"], transform=eval_tf, allowed_labels=allowed_labels)

    if config.get("USE_WEIGHTED_SAMPLER", False) and allowed_labels is None:
        sampler = make_sampler(
            train_ds,
            sampler_power=config["SAMPLER_POWER"],
            grade1_extra=config["GRADE1_EXTRA_WEIGHT"],
        )
        train_loader = DataLoader(
            train_ds,
            batch_size=config["BATCH_SIZE"],
            sampler=sampler,
            num_workers=config["NUM_WORKERS"],
            pin_memory=True,
            drop_last=True,
        )
    else:
        train_loader = DataLoader(
            train_ds,
            batch_size=config["BATCH_SIZE"],
            shuffle=shuffle_train,
            num_workers=config["NUM_WORKERS"],
            pin_memory=True,
            drop_last=True,
        )

    val_loader = DataLoader(
        val_ds,
        batch_size=max(1, config["BATCH_SIZE"] * 2),
        shuffle=False,
        num_workers=config["NUM_WORKERS"],
        pin_memory=True,
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=max(1, config["BATCH_SIZE"] * 2),
        shuffle=False,
        num_workers=config["NUM_WORKERS"],
        pin_memory=True,
    )
    return train_ds, val_ds, test_ds, train_loader, val_loader, test_loader

train_ds, val_ds, test_ds, train_loader, val_loader, test_loader = build_loaders(CONFIG)
print("Train/Val/Test:", len(train_ds), len(val_ds), len(test_ds))
print("Train counts:", class_counts_from_dataset(train_ds))


In [ ]:
# =========================
# 5) METRICS
# =========================

def safe_div(a, b):
    return float(a) / float(b) if b else 0.0

def compute_metrics(y_true, y_pred, probs=None, prefix=""):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    labels = np.arange(5)

    acc = accuracy_score(y_true, y_pred)
    bacc = balanced_accuracy_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    qwk = cohen_kappa_score(y_true, y_pred, weights="quadratic")
    prec, rec, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=labels, zero_division=0
    )

    out = {
        prefix + "acc": acc,
        prefix + "bacc": bacc,
        prefix + "mae": mae,
        prefix + "qwk": qwk,
        prefix + "macro_f1": float(np.mean(f1)),
        prefix + "g1_recall": float(rec[1]),
        prefix + "g1_precision": float(prec[1]),
        prefix + "g1_f1": float(f1[1]),
    }
    for g in labels:
        out[prefix + f"recall_g{g}"] = float(rec[g])
        out[prefix + f"precision_g{g}"] = float(prec[g])
        out[prefix + f"f1_g{g}"] = float(f1[g])
        out[prefix + f"support_g{g}"] = int(support[g])
    return out

def val_selection_score(metrics, config):
    return (
        config["VAL_SCORE_ACC_W"] * metrics["acc"]
        + config["VAL_SCORE_QWK_W"] * metrics["qwk"]
        + config["VAL_SCORE_G1_W"] * metrics["g1_recall"]
    )

def print_metrics(m, title="metrics"):
    keys = ["acc", "qwk", "mae", "bacc", "macro_f1", "g1_recall", "g1_precision", "g1_f1"]
    print("\n" + title)
    for k in keys:
        if k in m:
            print(f"  {k:14s}: {m[k]:.4f}")

def per_class_table(y_true, y_pred):
    labels = np.arange(5)
    prec, rec, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=labels, zero_division=0
    )
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    rows = []
    for g in labels:
        main_conf = ""
        errors = cm[g].copy()
        errors[g] = 0
        if errors.sum() > 0:
            j = int(np.argmax(errors))
            main_conf = f"Grade {j} ({errors[j]} / {errors.sum()} errors)"
        rows.append({
            "grade": g,
            "n": int(support[g]),
            "recall": rec[g],
            "precision": prec[g],
            "f1": f1[g],
            "main_confusion": main_conf,
        })
    return pd.DataFrame(rows)

def bootstrap_ci(y_true, y_pred, n_boot=1000, seed=42):
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    n = len(y_true)
    vals = defaultdict(list)
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        m = compute_metrics(y_true[idx], y_pred[idx])
        for k in ["acc", "qwk", "mae", "g1_recall", "macro_f1"]:
            vals[k].append(m[k])
    rows = []
    for k, arr in vals.items():
        arr = np.asarray(arr)
        rows.append({
            "metric": k,
            "mean": float(np.mean(arr)),
            "ci_low": float(np.percentile(arr, 2.5)),
            "ci_high": float(np.percentile(arr, 97.5)),
        })
    return pd.DataFrame(rows)

In [ ]:
# =========================
# 6) LOSSES
# =========================

def class_balanced_weights(counts, beta=0.999):
    counts = np.asarray(counts, dtype=np.float64)
    effective_num = 1.0 - np.power(beta, counts)
    weights = (1.0 - beta) / np.maximum(effective_num, 1e-12)
    weights = weights / weights.sum() * len(counts)
    return weights.astype(np.float32)

def inverse_sqrt_weights(counts):
    counts = np.asarray(counts, dtype=np.float64)
    w = 1.0 / np.sqrt(np.maximum(counts, 1))
    w = w / w.sum() * len(counts)
    return w.astype(np.float32)

class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=1.5, label_smoothing=0.0):
        super().__init__()
        self.register_buffer("weight", None if weight is None else torch.tensor(weight, dtype=torch.float32))
        self.gamma = gamma
        self.label_smoothing = label_smoothing

    def forward(self, logits, target):
        weight = self.weight.to(logits.device) if self.weight is not None else None
        ce = F.cross_entropy(
            logits, target, weight=weight, reduction="none",
            label_smoothing=self.label_smoothing
        )
        pt = torch.exp(-ce.detach())
        loss = ((1.0 - pt) ** self.gamma) * ce
        return loss.mean()

def cumulative_targets(labels, num_classes=5):
    # q_t = 1[y > t], t=0..K-2
    thresholds = torch.arange(num_classes - 1, device=labels.device).view(1, -1)
    return (labels.view(-1, 1) > thresholds).float()

def coral_bce_loss(coral_logits, labels, pos_weight=None, num_classes=5):
    q = cumulative_targets(labels, num_classes=num_classes)
    if pos_weight is not None:
        pos_weight = pos_weight.to(coral_logits.device)
    return F.binary_cross_entropy_with_logits(coral_logits, q, pos_weight=pos_weight)

def threshold_pos_weights(labels, num_classes=5):
    y = torch.tensor(labels, dtype=torch.long)
    q = cumulative_targets(y, num_classes=num_classes)
    pos = q.sum(0)
    neg = q.shape[0] - pos
    return (neg / torch.clamp(pos, min=1.0)).float()

def pairwise_margin_ranking_loss(scores, labels, margin=1.0):
    # scores: larger means more severe
    labels = labels.view(-1)
    scores = scores.view(-1)
    ydiff = labels.view(-1,1) - labels.view(1,-1)
    mask = ydiff > 0
    if mask.sum() == 0:
        return scores.sum() * 0.0
    sdiff = scores.view(-1,1) - scores.view(1,-1)
    return F.relu(margin - sdiff)[mask].mean()

def build_ce_criterion(config, train_labels):
    counts = np.bincount(train_labels, minlength=5)
    mode = config["LOSS_MODE"]

    if mode == "ce":
        weight = None
        return nn.CrossEntropyLoss(label_smoothing=config["LABEL_SMOOTHING"]), weight

    if mode == "weighted_ce":
        weight = inverse_sqrt_weights(counts)
        return nn.CrossEntropyLoss(
            weight=torch.tensor(weight, dtype=torch.float32),
            label_smoothing=config["LABEL_SMOOTHING"],
        ), weight

    if mode == "focal":
        weight = inverse_sqrt_weights(counts)
        return FocalLoss(weight=weight, gamma=config["FOCAL_GAMMA"], label_smoothing=config["LABEL_SMOOTHING"]), weight

    if mode == "cb_focal":
        weight = class_balanced_weights(counts, beta=config["CB_BETA"])
        # Extra emphasis for Grade 1, the reviewer-critical failure class.
        weight[1] *= 1.20
        weight = weight / weight.sum() * len(weight)
        return FocalLoss(weight=weight, gamma=config["FOCAL_GAMMA"], label_smoothing=config["LABEL_SMOOTHING"]), weight

    raise ValueError(f"Unknown LOSS_MODE: {mode}")

def hybrid_loss(outputs, labels, ce_criterion, config, coral_pos_weight=None):
    cls_logits = outputs["cls_logits"]
    coral_logits = outputs["coral_logits"]
    ordinal_score = outputs["ordinal_score"]

    ce = ce_criterion(cls_logits, labels)
    cor = coral_bce_loss(coral_logits, labels, pos_weight=coral_pos_weight)
    rank = pairwise_margin_ranking_loss(
        ordinal_score, labels, margin=config["PAIRWISE_MARGIN"]
    )

    loss = (
        config["CE_WEIGHT"] * ce
        + config["ORDINAL_WEIGHT"] * cor
        + config["RANK_WEIGHT"] * rank
    )
    return loss, {"loss_ce": float(ce.detach().cpu()), "loss_coral": float(cor.detach().cpu()), "loss_rank": float(rank.detach().cpu())}

In [ ]:
# =========================
# 7) MODEL
# =========================

class SEBlock(nn.Module):
    def __init__(self, dim, reduction=16):
        super().__init__()
        hidden = max(dim // reduction, 16)
        self.net = nn.Sequential(
            nn.Linear(dim, hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, dim),
            nn.Sigmoid(),
        )
    def forward(self, x):
        return x * self.net(x)

def _safe_weights(weight_enum, pretrained=True):
    if not pretrained:
        return None
    try:
        return weight_enum.DEFAULT
    except Exception:
        return None

def create_backbone(name="efficientnet_b3", pretrained=True):
    name = name.lower()
    try:
        if name == "efficientnet_b0":
            weights = _safe_weights(models.EfficientNet_B0_Weights, pretrained)
            m = models.efficientnet_b0(weights=weights)
            dim = m.classifier[1].in_features
            m.classifier = nn.Identity()
            return m, dim

        if name == "efficientnet_b3":
            weights = _safe_weights(models.EfficientNet_B3_Weights, pretrained)
            m = models.efficientnet_b3(weights=weights)
            dim = m.classifier[1].in_features
            m.classifier = nn.Identity()
            return m, dim

        if name == "efficientnet_b4":
            weights = _safe_weights(models.EfficientNet_B4_Weights, pretrained)
            m = models.efficientnet_b4(weights=weights)
            dim = m.classifier[1].in_features
            m.classifier = nn.Identity()
            return m, dim

        if name == "efficientnet_v2_s":
            weights = _safe_weights(models.EfficientNet_V2_S_Weights, pretrained)
            m = models.efficientnet_v2_s(weights=weights)
            dim = m.classifier[1].in_features
            m.classifier = nn.Identity()
            return m, dim

        if name == "convnext_tiny":
            weights = _safe_weights(models.ConvNeXt_Tiny_Weights, pretrained)
            m = models.convnext_tiny(weights=weights)
            dim = m.classifier[2].in_features
            m.classifier = nn.Identity()
            return m, dim

        if name == "resnet50":
            weights = _safe_weights(models.ResNet50_Weights, pretrained)
            m = models.resnet50(weights=weights)
            dim = m.fc.in_features
            m.fc = nn.Identity()
            return m, dim

    except Exception as e:
        print("Pretrained model loading failed, retrying with random weights.")
        print("Error:", repr(e))
        return create_backbone(name=name, pretrained=False)

    raise ValueError(f"Unsupported backbone: {name}")

class OAHybridNet(nn.Module):
    def __init__(self, backbone_name="efficientnet_b3", pretrained=True, num_classes=5, use_extra_se=True, dropout=0.35):
        super().__init__()
        self.num_classes = num_classes
        self.backbone, dim = create_backbone(backbone_name, pretrained=pretrained)
        self.feature_dim = dim
        self.use_extra_se = use_extra_se
        self.extra_se = SEBlock(dim, reduction=16) if use_extra_se else nn.Identity()
        self.dropout = nn.Dropout(dropout)

        self.cls_head = nn.Linear(dim, num_classes)

        # Ordered CORAL-style cutpoints:
        # logit_t = score - cutpoint_t, with cutpoint_0 < cutpoint_1 < ... guaranteed by softplus deltas.
        self.ordinal_score_head = nn.Linear(dim, 1)
        init_cutpoints = torch.linspace(-1.5, 1.5, steps=num_classes - 1)
        self.cutpoint_first = nn.Parameter(init_cutpoints[:1].clone())
        init_deltas = init_cutpoints[1:] - init_cutpoints[:-1]
        self.cutpoint_raw_deltas = nn.Parameter(torch.log(torch.expm1(init_deltas) + 1e-6))

    def ordered_cutpoints(self):
        if self.num_classes <= 2:
            return self.cutpoint_first
        deltas = F.softplus(self.cutpoint_raw_deltas)
        rest = self.cutpoint_first + torch.cumsum(deltas, dim=0)
        return torch.cat([self.cutpoint_first, rest], dim=0)

    def forward(self, x):
        feat = self.backbone(x)
        if feat.ndim > 2:
            feat = torch.flatten(feat, 1)
        feat = self.extra_se(feat)
        feat = self.dropout(feat)

        cls_logits = self.cls_head(feat)
        score = self.ordinal_score_head(feat)  # [B,1]
        cutpoints = self.ordered_cutpoints().view(1, -1)
        coral_logits = score - cutpoints
        ordinal_score = torch.sigmoid(coral_logits).sum(dim=1)

        return {
            "features": feat,
            "cls_logits": cls_logits,
            "coral_logits": coral_logits,
            "ordinal_score": ordinal_score,
            "cutpoints": cutpoints.detach(),
        }

def coral_logits_to_probs(coral_logits):
    # For K=5: P(y=0)=1-P(y>0), P(y=1)=P(y>0)-P(y>1), ..., P(y=4)=P(y>3)
    gt = torch.sigmoid(coral_logits)
    probs = torch.cat([
        1.0 - gt[:, 0:1],
        gt[:, 0:1] - gt[:, 1:2],
        gt[:, 1:2] - gt[:, 2:3],
        gt[:, 2:3] - gt[:, 3:4],
        gt[:, 3:4],
    ], dim=1)
    probs = probs.clamp(min=1e-7)
    probs = probs / probs.sum(dim=1, keepdim=True)
    return probs

def combined_probs(outputs, ordinal_blend=0.25):
    cls_probs = F.softmax(outputs["cls_logits"], dim=1)
    ord_probs = coral_logits_to_probs(outputs["coral_logits"])
    probs = (1.0 - ordinal_blend) * cls_probs + ordinal_blend * ord_probs
    probs = probs / probs.sum(dim=1, keepdim=True)
    return probs

In [ ]:
# =========================
# 8) TRAIN / EVALUATE FUNCTIONS
# =========================

def make_optimizer(model, config):
    backbone_params, head_params = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if name.startswith("backbone."):
            backbone_params.append(p)
        else:
            head_params.append(p)
    return torch.optim.AdamW(
        [
            {"params": backbone_params, "lr": config["LR_BACKBONE"]},
            {"params": head_params, "lr": config["LR_HEAD"]},
        ],
        weight_decay=config["WEIGHT_DECAY"]
    )

@torch.no_grad()
def run_inference(model, loader, config, use_tta_hflip=False):
    model.eval()
    all_y, all_idx, all_probs, all_cls_logits = [], [], [], []

    for x, y, idx in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        outputs = model(x)
        probs = combined_probs(outputs, ordinal_blend=config["ORDINAL_PROB_BLEND"])
        cls_logits = outputs["cls_logits"]

        if use_tta_hflip:
            xf = torch.flip(x, dims=[3])
            out_f = model(xf)
            probs_f = combined_probs(out_f, ordinal_blend=config["ORDINAL_PROB_BLEND"])
            cls_logits_f = out_f["cls_logits"]
            probs = 0.5 * (probs + probs_f)
            cls_logits = 0.5 * (cls_logits + cls_logits_f)

        all_y.append(y.cpu().numpy())
        all_idx.append(idx.numpy())
        all_probs.append(probs.cpu().numpy())
        all_cls_logits.append(cls_logits.cpu().numpy())

    y_true = np.concatenate(all_y)
    indices = np.concatenate(all_idx)
    probs = np.concatenate(all_probs)
    cls_logits = np.concatenate(all_cls_logits)
    y_pred = probs.argmax(axis=1)

    return {
        "y_true": y_true,
        "y_pred": y_pred,
        "probs": probs,
        "cls_logits": cls_logits,
        "indices": indices,
        "metrics": compute_metrics(y_true, y_pred, probs=probs),
    }

def train_one_seed(config, seed=42, experiment_name="main"):
    set_seed(seed)
    out_dir = OUT_DIR / experiment_name / f"seed_{seed}"
    out_dir.mkdir(parents=True, exist_ok=True)
    with open(out_dir / "config_used.json", "w") as f:
        json.dump({k: (str(v) if isinstance(v, Path) else v) for k, v in config.items()}, f, indent=2)

    train_ds, val_ds, test_ds, train_loader, val_loader, test_loader = build_loaders(config)
    train_counts = class_counts_from_dataset(train_ds)

    model = OAHybridNet(
        backbone_name=config["BACKBONE"],
        pretrained=config["PRETRAINED"],
        use_extra_se=config["USE_EXTRA_SE"],
        dropout=config.get("DROPOUT", 0.35),
    ).to(DEVICE)

    ce_criterion, ce_weight = build_ce_criterion(config, train_ds.labels)
    ce_criterion = ce_criterion.to(DEVICE)
    coral_pos_weight = threshold_pos_weights(train_ds.labels, num_classes=5).to(DEVICE)

    optimizer = make_optimizer(model, config)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=config["EPOCHS"], eta_min=1e-6
    )
    scaler = torch.cuda.amp.GradScaler(enabled=(config["USE_AMP"] and DEVICE.type == "cuda"))

    best_score = -1e9
    best_state = None
    best_epoch = -1
    patience_left = config["PATIENCE"]
    history = []

    for epoch in range(1, config["EPOCHS"] + 1):
        model.train()
        t0 = time.time()
        running = defaultdict(float)
        n_batches = 0

        for x, y, idx in train_loader:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=(config["USE_AMP"] and DEVICE.type == "cuda")):
                outputs = model(x)
                loss, parts = hybrid_loss(outputs, y, ce_criterion, config, coral_pos_weight=coral_pos_weight)

            scaler.scale(loss).backward()
            if config["GRAD_CLIP"] is not None:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), config["GRAD_CLIP"])
            scaler.step(optimizer)
            scaler.update()

            running["loss"] += float(loss.detach().cpu())
            for k, v in parts.items():
                running[k] += v
            n_batches += 1

        scheduler.step()

        val_out = run_inference(model, val_loader, config, use_tta_hflip=False)
        val_m = val_out["metrics"]
        score = val_selection_score(val_m, config)

        row = {
            "epoch": epoch,
            "seed": seed,
            "lr_backbone": optimizer.param_groups[0]["lr"],
            "lr_head": optimizer.param_groups[1]["lr"],
            "train_loss": running["loss"] / max(n_batches, 1),
            "train_loss_ce": running["loss_ce"] / max(n_batches, 1),
            "train_loss_coral": running["loss_coral"] / max(n_batches, 1),
            "train_loss_rank": running["loss_rank"] / max(n_batches, 1),
            "val_score": score,
            **{f"val_{k}": v for k, v in val_m.items()},
            "seconds": time.time() - t0,
        }
        history.append(row)

        improved = score > best_score
        if improved:
            best_score = score
            best_epoch = epoch
            # Store the best state on CPU to avoid accumulating large GPU copies across epochs/seeds.
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            if config.get("SAVE_CHECKPOINTS", False):
                torch.save({
                    "model": best_state,
                    "config": config,
                    "seed": seed,
                    "epoch": epoch,
                    "val_metrics": val_m,
                    "val_score": score,
                }, out_dir / "best_model.pt")
            patience_left = config["PATIENCE"]
        else:
            patience_left -= 1

        print(
            f"[{experiment_name} seed={seed}] epoch {epoch:03d}/{config['EPOCHS']} "
            f"loss={row['train_loss']:.4f} val_acc={val_m['acc']:.4f} "
            f"val_qwk={val_m['qwk']:.4f} val_g1rec={val_m['g1_recall']:.4f} "
            f"score={score:.4f} {'*' if improved else ''}"
        )

        if patience_left <= 0:
            print(f"Early stopping at epoch {epoch}. Best epoch: {best_epoch}")
            break

    hist_df = pd.DataFrame(history)
    hist_df.to_csv(out_dir / "history.csv", index=False)

    model.load_state_dict(best_state)

    val_final = run_inference(model, val_loader, config, use_tta_hflip=config["USE_TTA_HFLIP"])
    test_final = run_inference(model, test_loader, config, use_tta_hflip=config["USE_TTA_HFLIP"])

    # Save predictions
    for split_name, out in [("val", val_final), ("test", test_final)]:
        pred_df = pd.DataFrame({
            "idx": out["indices"],
            "y_true": out["y_true"],
            "y_pred": out["y_pred"],
        })
        for c in range(5):
            pred_df[f"prob_{c}"] = out["probs"][:, c]
            pred_df[f"cls_logit_{c}"] = out["cls_logits"][:, c]
        pred_df.to_csv(out_dir / f"{split_name}_predictions.csv", index=False)

        cm = confusion_matrix(out["y_true"], out["y_pred"], labels=np.arange(5))
        pd.DataFrame(cm).to_csv(out_dir / f"{split_name}_confusion_matrix.csv", index=False)
        per_class_table(out["y_true"], out["y_pred"]).to_csv(out_dir / f"{split_name}_per_class.csv", index=False)

    summary = {
        "experiment": experiment_name,
        "seed": seed,
        "best_epoch": best_epoch,
        **{f"val_{k}": v for k, v in val_final["metrics"].items()},
        **{f"test_{k}": v for k, v in test_final["metrics"].items()},
    }
    pd.DataFrame([summary]).to_csv(out_dir / "summary.csv", index=False)

    print_metrics(val_final["metrics"], title=f"VAL final seed={seed}")
    print_metrics(test_final["metrics"], title=f"TEST final seed={seed}")

    # Keep the returned model on CPU so multiple seeds do not occupy GPU memory.
    model.to("cpu")
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    return {
        "model": model,
        "summary": summary,
        "history": hist_df,
        "val": val_final,
        "test": test_final,
        "out_dir": out_dir,
    }

In [ ]:
# =========================
# 9) CLASS-BIAS TUNING ON VALIDATION
# =========================

def apply_class_bias(probs, bias):
    # bias is additive in log-prob space.
    logp = np.log(np.clip(probs, 1e-12, 1.0))
    biased = logp + bias.reshape(1, -1)
    return biased.argmax(axis=1)

def tune_class_bias(val_probs, y_val, config, verbose=True):
    base_pred = val_probs.argmax(axis=1)
    base_m = compute_metrics(y_val, base_pred)
    base_acc = base_m["acc"]

    # Search mainly class 1, also class 0/2 to manage boundary trade-off.
    # Wider class-1 bias improves Grade-1 recall but may harm accuracy.
    # v2: conservative bias search. Do not sacrifice the newly improved accuracy.
    b0_grid = np.round(np.arange(-0.30, 0.31, 0.10), 2)
    b1_grid = np.round(np.arange(-0.10, 1.21, 0.05), 2)
    b2_grid = np.round(np.arange(-0.30, 0.31, 0.10), 2)

    best = {
        "score": -1e9,
        "bias": np.zeros(5, dtype=np.float32),
        "metrics": base_m,
    }

    # Penalize configurations that destroy exact accuracy.
    # This prevents "Grade-1 recall went up but accuracy collapsed" solutions.
    # Accuracy-first: allow at most a tiny validation accuracy drop.
    min_acc = max(0.0, base_acc - 0.003)

    for b0 in b0_grid:
        for b1 in b1_grid:
            for b2 in b2_grid:
                bias = np.array([b0, b1, b2, 0.0, 0.0], dtype=np.float32)
                pred = apply_class_bias(val_probs, bias)
                m = compute_metrics(y_val, pred)
                penalty = max(0.0, min_acc - m["acc"]) * 4.0
                score = (
                    0.80 * m["acc"]
                    + 0.15 * m["qwk"]
                    + 0.05 * m["g1_recall"]
                    - penalty
                )
                if score > best["score"]:
                    best = {"score": score, "bias": bias, "metrics": m}

    if verbose:
        print_metrics(base_m, "Validation before class-bias tuning")
        print("Best bias:", best["bias"])
        print_metrics(best["metrics"], "Validation after class-bias tuning")
    return best

def evaluate_with_bias(out, bias):
    pred = apply_class_bias(out["probs"], bias)
    m = compute_metrics(out["y_true"], pred)
    return pred, m

def run_bias_tuning_and_save(seed_results, experiment_name="main_bias_tuned"):
    rows = []
    for r in seed_results:
        seed = r["summary"]["seed"]
        best = tune_class_bias(r["val"]["probs"], r["val"]["y_true"], CONFIG, verbose=True)
        val_pred, val_m = evaluate_with_bias(r["val"], best["bias"])
        test_pred, test_m = evaluate_with_bias(r["test"], best["bias"])

        out_dir = r["out_dir"]
        np.save(out_dir / "best_class_bias.npy", best["bias"])

        pd.DataFrame({
            "idx": r["test"]["indices"],
            "y_true": r["test"]["y_true"],
            "y_pred_before": r["test"]["y_pred"],
            "y_pred_after_bias": test_pred,
        }).to_csv(out_dir / "test_predictions_bias_tuned.csv", index=False)

        per_class_table(r["test"]["y_true"], test_pred).to_csv(out_dir / "test_per_class_bias_tuned.csv", index=False)
        pd.DataFrame(confusion_matrix(r["test"]["y_true"], test_pred, labels=np.arange(5))).to_csv(
            out_dir / "test_confusion_matrix_bias_tuned.csv", index=False
        )

        row = {
            "experiment": experiment_name,
            "seed": seed,
            **{f"val_{k}": v for k, v in val_m.items()},
            **{f"test_{k}": v for k, v in test_m.items()},
            "bias": json.dumps(best["bias"].tolist()),
        }
        rows.append(row)
        print_metrics(test_m, title=f"TEST after class-bias tuning seed={seed}")
    df = pd.DataFrame(rows)
    summary_dir = OUT_DIR / experiment_name
    summary_dir.mkdir(parents=True, exist_ok=True)
    df.to_csv(summary_dir / "summary_bias_tuned.csv", index=False)
    return df

## Çalıştırma stratejisi — v6

Bu notebook, **ana 3-seed CE/OA koşusunu tekrar etmez**. Önceki v5 çıktılarını şu Kaggle input path’inden içeri alır:

```text
/kaggle/input/<anonymous_previous_outputs>/previous-main-outputs
```

Sonra sadece eksik kalan hakem kapatma deneylerini çalıştırır:

```python
CONFIG["RUN_MAIN_EXPERIMENTS"] = False
CONFIG["IMPORT_PREVIOUS_OUTPUTS"] = True
CONFIG["RUN_FULL_ABLATION"] = True
CONFIG["RUN_LAMBDA_DELTA_SENSITIVITY"] = True
CONFIG["REVIEWER_TABLE_SEEDS"] = [42]
```

Varsayılan tek-seed reviewer tabloları hesaplama maliyetini azaltmak içindir. Ana CE/OA karşılaştırması zaten v5 çıktılarından 3 seed olarak korunur. Çok daha güçlü reviewer tabloları istenirse şunu yap:

```python
CONFIG["REVIEWER_TABLE_SEEDS"] = [42, 7, 123]
```

Ama bu B4/380 için çok uzun sürebilir.


In [ ]:
# =========================
# 10) RUN MAIN EXPERIMENTS — CE baseline + moderate ordinal model, same backbone
# =========================

main_results_by_experiment = {}

if CONFIG.get("RUN_MAIN_EXPERIMENTS", True):
    for exp in MAIN_EXPERIMENTS:
        exp_name = exp["name"]
        exp_cfg = copy.deepcopy(CONFIG)
        exp_cfg.update({k: v for k, v in exp.items() if k != "name"})

        seed_results = []
        for seed in exp_cfg["SEEDS"]:
            res = train_one_seed(exp_cfg, seed=seed, experiment_name=exp_name)
            seed_results.append(res)
        main_results_by_experiment[exp_name] = seed_results

        summary_df = pd.DataFrame([r["summary"] for r in seed_results])
        summary_dir = OUT_DIR / exp_name
        summary_dir.mkdir(parents=True, exist_ok=True)
        summary_df.to_csv(summary_dir / "summary_all_seeds.csv", index=False)
        display(summary_df)

        mean_row = summary_df.mean(numeric_only=True).to_frame("mean").T
        std_row = summary_df.std(numeric_only=True).to_frame("std").T
        mean_std_df = pd.concat([mean_row, std_row], axis=0)
        mean_std_df.to_csv(summary_dir / "summary_all_seeds_mean_std.csv", index=True)
        display(mean_std_df)

    ce_results = main_results_by_experiment[MAIN_EXPERIMENTS[0]["name"]]
    ordinal_model_results = main_results_by_experiment[MAIN_EXPERIMENTS[1]["name"]]
    main_results = ordinal_model_results  # default for backward-compatible optional cells

    print("\nMain experiments completed.")
    print("Now compare CE vs moderate ordinal model using:")
    print(" - summary_all_seeds_mean_std.csv")
    print(" - significance_test_summary_raw.csv / significance_test_summary_bias_tuned.csv")
    print(" - ensemble_summary.csv files")
else:
    print("CONFIG['RUN_MAIN_EXPERIMENTS'] is False; skipped main experiments.")


In [ ]:
# =========================
# 11) VALIDATION-TUNED CLASS BIAS — both arms
# =========================
# Selected on validation predictions only, then applied to test predictions.
# This is acceptable as long as the bias is chosen using validation only and both arms are treated identically.

bias_summary_by_experiment = {}

if CONFIG["RUN_CLASS_BIAS_TUNING"] and "main_results_by_experiment" in globals() and len(main_results_by_experiment):
    for exp_name, seed_results in main_results_by_experiment.items():
        bias_exp_name = f"{exp_name}_bias_tuned"
        bias_df = run_bias_tuning_and_save(seed_results, experiment_name=bias_exp_name)
        bias_summary_by_experiment[exp_name] = bias_df
        display(bias_df)

        bias_summary_dir = OUT_DIR / bias_exp_name
        bias_summary_dir.mkdir(parents=True, exist_ok=True)
        mean_row = bias_df.mean(numeric_only=True).to_frame("mean").T
        std_row = bias_df.std(numeric_only=True).to_frame("std").T
        bias_mean_std_df = pd.concat([mean_row, std_row], axis=0)
        bias_mean_std_df.to_csv(bias_summary_dir / "summary_bias_tuned_mean_std.csv", index=True)
        display(bias_mean_std_df)
else:
    print("Bias tuning skipped or main experiments not available.")


In [ ]:
# =========================
# 11B) SIGNIFICANCE TEST: CE baseline vs moderate ordinal model
# =========================
# Reviewer-2 asked whether the reported improvement is statistically supported.
# This cell creates:
#   - seed-paired descriptive tests
#   - sample-level paired bootstrap for raw ensemble predictions
#   - sample-level paired bootstrap for validation-bias-tuned ensemble predictions
#
# With n=3 seeds, seed-level Wilcoxon is underpowered. Treat it as descriptive.
# The sample-level paired bootstrap is more informative for the held-out test set.

try:
    from scipy import stats
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False
    print("scipy not available; Wilcoxon will be skipped.")

def bias_probs(probs, bias):
    logp = np.log(np.clip(probs, 1e-12, 1.0)) + bias.reshape(1, -1)
    out = np.exp(logp - logp.max(axis=1, keepdims=True))
    return out / out.sum(axis=1, keepdims=True)

def ensemble_probs_from_seeds(seed_results, use_saved_bias=False):
    y_true = seed_results[0]["test"]["y_true"]
    assert all(np.array_equal(y_true, r["test"]["y_true"]) for r in seed_results)

    probs_list = []
    for r in seed_results:
        probs = r["test"]["probs"].copy()
        if use_saved_bias:
            bias_path = r["out_dir"] / "best_class_bias.npy"
            if bias_path.exists():
                probs = bias_probs(probs, np.load(bias_path))
        probs_list.append(probs)
    mean_probs = np.mean(probs_list, axis=0)
    return y_true, mean_probs

def paired_bootstrap_metric_diff(y_true, probs_a, probs_b, metric_fn, n_boot=2000, seed=0):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    diffs = np.empty(n_boot)
    pred_a_full = probs_a.argmax(axis=1)
    pred_b_full = probs_b.argmax(axis=1)
    for i in range(n_boot):
        idx = rng.integers(0, n, n)
        diffs[i] = metric_fn(y_true[idx], pred_b_full[idx]) - metric_fn(y_true[idx], pred_a_full[idx])
    return diffs

def qwk_metric(y_true, y_pred):
    return compute_metrics(y_true, y_pred)["qwk"]

def acc_metric(y_true, y_pred):
    return compute_metrics(y_true, y_pred)["acc"]

def neg_mae_metric(y_true, y_pred):
    return -compute_metrics(y_true, y_pred)["mae"]  # higher is better

def macro_f1_metric(y_true, y_pred):
    return compute_metrics(y_true, y_pred)["macro_f1"]

def g1_recall_metric(y_true, y_pred):
    return compute_metrics(y_true, y_pred)["g1_recall"]

def run_significance_block(ce_results, oa_results, use_saved_bias=False, label="raw"):
    rows_seed = []
    rows_boot_summary = []
    boot_dir = OUT_DIR / "significance_tests"
    boot_dir.mkdir(parents=True, exist_ok=True)

    # --- Seed-paired descriptive comparison ---
    for metric_key in ["test_qwk", "test_acc", "test_mae", "test_macro_f1", "test_g1_recall"]:
        ce_vals = np.array([r["summary"][metric_key] for r in ce_results], dtype=float)
        oa_vals = np.array([r["summary"][metric_key] for r in oa_results], dtype=float)
        diff = oa_vals - ce_vals
        row = {
            "variant": label,
            "metric": metric_key,
            "baseline_values": json.dumps(ce_vals.tolist()),
            "ordinal_model_values": json.dumps(oa_vals.tolist()),
            "diff_mean": float(diff.mean()),
            "diff_std": float(diff.std(ddof=1)) if len(diff) > 1 else 0.0,
        }
        if SCIPY_AVAILABLE and len(diff) >= 3:
            try:
                _, p = stats.wilcoxon(oa_vals, ce_vals)
                row["wilcoxon_p"] = float(p)
            except Exception as e:
                row["wilcoxon_error"] = repr(e)
        rows_seed.append(row)

    # --- Sample-level paired bootstrap on seed ensemble ---
    y_true, probs_ce = ensemble_probs_from_seeds(ce_results, use_saved_bias=use_saved_bias)
    _, probs_oa = ensemble_probs_from_seeds(oa_results, use_saved_bias=use_saved_bias)

    for name, metric_fn in [
        ("qwk", qwk_metric),
        ("acc", acc_metric),
        ("neg_mae", neg_mae_metric),
        ("macro_f1", macro_f1_metric),
        ("g1_recall", g1_recall_metric),
    ]:
        diffs = paired_bootstrap_metric_diff(y_true, probs_ce, probs_oa, metric_fn, n_boot=2000, seed=42)
        ci_lo, ci_hi = np.percentile(diffs, [2.5, 97.5])
        p_no_improvement = float((diffs <= 0).mean())
        pd.DataFrame({"diff_oa_minus_baseline": diffs}).to_csv(
            boot_dir / f"bootstrap_diffs_{label}_{name}.csv", index=False
        )
        rows_boot_summary.append({
            "variant": label,
            "metric": name,
            "mean_diff_oa_minus_baseline": float(diffs.mean()),
            "ci95_low": float(ci_lo),
            "ci95_high": float(ci_hi),
            "p_no_improvement": p_no_improvement,
            "n_boot": int(len(diffs)),
        })
        print(f"{label} {name}: mean diff={diffs.mean():.4f}, 95% CI=[{ci_lo:.4f}, {ci_hi:.4f}], P(no improvement)={p_no_improvement:.3f}")

    seed_df = pd.DataFrame(rows_seed)
    boot_df = pd.DataFrame(rows_boot_summary)
    seed_df.to_csv(boot_dir / f"seed_paired_summary_{label}.csv", index=False)
    boot_df.to_csv(boot_dir / f"bootstrap_summary_{label}.csv", index=False)
    display(seed_df)
    display(boot_df)
    return seed_df, boot_df

if "ce_results" in globals() and "ordinal_model_results" in globals():
    seed_raw_df, boot_raw_df = run_significance_block(ce_results, ordinal_model_results, use_saved_bias=False, label="raw")
    if CONFIG.get("RUN_CLASS_BIAS_TUNING", False):
        seed_bias_df, boot_bias_df = run_significance_block(ce_results, ordinal_model_results, use_saved_bias=True, label="bias_tuned")
else:
    print("Run the main-experiments cell first.")


In [ ]:
# =========================
# 12) ENSEMBLE ACROSS SEEDS — saved separately for each arm
# =========================
# Uses the mean predicted probability across seeds.
# Both CE baseline and moderate ordinal model are ensembled so the comparison remains fair.

def ensemble_seed_outputs(results, split="test", use_bias=False):
    y_true = results[0][split]["y_true"]
    assert all(np.array_equal(y_true, r[split]["y_true"]) for r in results)

    probs_list = []
    for r in results:
        probs = r[split]["probs"].copy()
        if use_bias:
            bias_path = r["out_dir"] / "best_class_bias.npy"
            if bias_path.exists():
                probs = bias_probs(probs, np.load(bias_path))
        probs_list.append(probs)

    ens_probs = np.mean(probs_list, axis=0)
    pred = ens_probs.argmax(axis=1)
    m = compute_metrics(y_true, pred, probs=ens_probs)
    return y_true, pred, ens_probs, m

ensemble_rows = []

if "main_results_by_experiment" in globals() and len(main_results_by_experiment):
    for exp_name, results in main_results_by_experiment.items():
        ens_dir = OUT_DIR / exp_name
        ens_dir.mkdir(parents=True, exist_ok=True)

        for use_bias, suffix, label in [
            (False, "", "ensemble_no_bias"),
            (True, "_bias", "ensemble_with_validation_bias"),
        ]:
            if use_bias and not CONFIG.get("RUN_CLASS_BIAS_TUNING", False):
                continue

            for split in ["val", "test"]:
                y_true, pred, probs, m = ensemble_seed_outputs(results, split=split, use_bias=use_bias)
                print_metrics(m, title=f"{exp_name} | {split.upper()} | {label}")

                pd.DataFrame({"y_true": y_true, "y_pred": pred}).to_csv(
                    ens_dir / f"{split}_ensemble{suffix}_predictions.csv", index=False
                )
                prob_df = pd.DataFrame(probs, columns=[f"prob_{i}" for i in range(5)])
                prob_df.insert(0, "y_pred", pred)
                prob_df.insert(0, "y_true", y_true)
                prob_df.to_csv(ens_dir / f"{split}_ensemble{suffix}_predictions_with_probs.csv", index=False)

                per_class_table(y_true, pred).to_csv(
                    ens_dir / f"{split}_ensemble{suffix}_per_class.csv", index=False
                )
                pd.DataFrame(confusion_matrix(y_true, pred, labels=np.arange(5))).to_csv(
                    ens_dir / f"{split}_ensemble{suffix}_confusion_matrix.csv", index=False
                )

                ensemble_rows.append({"experiment": exp_name, "split": split, "variant": label, **m})

ensemble_summary_df = pd.DataFrame(ensemble_rows)
if len(ensemble_summary_df):
    ensemble_summary_df.to_csv(OUT_DIR / "ensemble_summary_all_experiments.csv", index=False)
    display(ensemble_summary_df)
else:
    print("No ensemble outputs generated yet.")


In [ ]:
# =========================
# 12B) OPTIONAL MODEL-ZOO QUICK SWEEP
# =========================
# Turn CONFIG["RUN_MODEL_ZOO_SWEEP"] = True in the first cell, then run this cell.
# It trains several stronger backbones for a short probe and ranks them by validation accuracy.
# After the best candidate is identified, set CONFIG accordingly and run the main experiment with 3 seeds.

def run_model_zoo_sweep():
    rows = []
    for item in CONFIG["MODEL_ZOO_SWEEP"]:
        cfg = copy.deepcopy(CONFIG)
        cfg.update(item)
        cfg["SEEDS"] = [CONFIG["SWEEP_SEED"]]
        cfg["EPOCHS"] = CONFIG["SWEEP_EPOCHS"]
        cfg["PATIENCE"] = CONFIG["SWEEP_PATIENCE"]
        cfg["RUN_CLASS_BIAS_TUNING"] = False
        name = f"sweep_{item['name']}"
        print("\n" + "="*80)
        print("Running", name, "with", {k: cfg[k] for k in ["BACKBONE", "IMG_SIZE", "BATCH_SIZE", "DROPOUT"]})
        print("="*80)
        try:
            res = train_one_seed(cfg, seed=CONFIG["SWEEP_SEED"], experiment_name=name)
            row = {
                "sweep_name": item["name"],
                "backbone": cfg["BACKBONE"],
                "img_size": cfg["IMG_SIZE"],
                "batch_size": cfg["BATCH_SIZE"],
                **res["summary"],
            }
            rows.append(row)
        except RuntimeError as e:
            print("FAILED:", repr(e))
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()
            rows.append({
                "sweep_name": item["name"],
                "backbone": cfg["BACKBONE"],
                "img_size": cfg["IMG_SIZE"],
                "batch_size": cfg["BATCH_SIZE"],
                "failed": True,
                "error": repr(e),
            })
    df = pd.DataFrame(rows)
    df = df.sort_values(["val_acc", "val_qwk"], ascending=False, na_position="last")
    df.to_csv(OUT_DIR / "model_zoo_sweep_summary.csv", index=False)
    display(df)
    return df

if CONFIG.get("RUN_MODEL_ZOO_SWEEP", False):
    sweep_df = run_model_zoo_sweep()
else:
    print("Model-zoo sweep is OFF. Set CONFIG['RUN_MODEL_ZOO_SWEEP']=True to run it.")


In [ ]:
# =========================
# 13) OPTIONAL GRADE-0/1 BOUNDARY REFINER
# =========================
# Purpose:
# - Hakemin en net vurduğu yer Grade-1 recall.
# - Base model 0 veya 1 tahmin ettiğinde, ayrıca 0-vs-1 uzmanı ile boundary correction yapılır.
# - Bu bölüm default kapalıdır. Önce ana 3-seed modeli bitirin, sonra açın.

class BinaryNet(nn.Module):
    def __init__(self, backbone_name="efficientnet_b0", pretrained=True, dropout=0.30):
        super().__init__()
        self.backbone, dim = create_backbone(backbone_name, pretrained=pretrained)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(dim, 2)
    def forward(self, x):
        feat = self.backbone(x)
        if feat.ndim > 2:
            feat = torch.flatten(feat, 1)
        return self.head(self.dropout(feat))

def build_binary_loaders(config):
    cfg = copy.deepcopy(config)
    cfg["USE_WEIGHTED_SAMPLER"] = False
    train_tf, eval_tf = build_transforms(cfg["IMG_SIZE"])
    train_ds = KneeOADataset(SPLITS["train"], transform=train_tf, allowed_labels=[0, 1])
    val_ds = KneeOADataset(SPLITS["val"], transform=eval_tf, allowed_labels=[0, 1])
    test_ds = KneeOADataset(SPLITS["test"], transform=eval_tf, allowed_labels=[0, 1])

    # remap labels: 0->0, 1->1 already OK.
    counts = class_counts_from_dataset(train_ds, num_classes=5)[:2]
    inv = 1.0 / np.maximum(counts, 1)
    weights_per_class = inv / inv.sum() * 2
    sample_weights = weights_per_class[train_ds.labels]
    sampler = WeightedRandomSampler(torch.DoubleTensor(sample_weights), num_samples=len(sample_weights), replacement=True)

    train_loader = DataLoader(train_ds, batch_size=cfg["BATCH_SIZE"], sampler=sampler, num_workers=cfg["NUM_WORKERS"], pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=cfg["BATCH_SIZE"]*2, shuffle=False, num_workers=cfg["NUM_WORKERS"], pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=cfg["BATCH_SIZE"]*2, shuffle=False, num_workers=cfg["NUM_WORKERS"], pin_memory=True)
    return train_ds, val_ds, test_ds, train_loader, val_loader, test_loader

def train_refiner(seed=42):
    set_seed(seed)
    train_ds_b, val_ds_b, test_ds_b, train_loader_b, val_loader_b, test_loader_b = build_binary_loaders(CONFIG)
    model = BinaryNet(CONFIG["REFINER_BACKBONE"], pretrained=CONFIG["PRETRAINED"]).to(DEVICE)
    counts = np.bincount(train_ds_b.labels, minlength=2)
    weight = torch.tensor(inverse_sqrt_weights(counts), dtype=torch.float32).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=weight, label_smoothing=0.02)
    opt = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CONFIG["REFINER_EPOCHS"], eta_min=1e-6)
    scaler = torch.cuda.amp.GradScaler(enabled=(CONFIG["USE_AMP"] and DEVICE.type == "cuda"))

    best_acc = -1
    best_state = None
    for epoch in range(1, CONFIG["REFINER_EPOCHS"]+1):
        model.train()
        losses = []
        for x,y,idx in train_loader_b:
            x = x.to(DEVICE); y = y.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=(CONFIG["USE_AMP"] and DEVICE.type == "cuda")):
                logits = model(x)
                loss = criterion(logits, y)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            losses.append(float(loss.detach().cpu()))
        sched.step()

        # eval on binary val
        out = infer_binary_refiner(model, val_loader_b)
        acc = accuracy_score(out["y_true"], out["y_pred"])
        print(f"[refiner seed={seed}] epoch={epoch:03d} loss={np.mean(losses):.4f} bin_val_acc={acc:.4f}")
        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    return model

@torch.no_grad()
def infer_binary_refiner(model, loader):
    model.eval()
    ys, idxs, probs = [], [], []
    for x,y,idx in loader:
        x = x.to(DEVICE)
        logits = model(x)
        p = F.softmax(logits, dim=1)
        if CONFIG["USE_TTA_HFLIP"]:
            pf = F.softmax(model(torch.flip(x, dims=[3])), dim=1)
            p = 0.5 * (p + pf)
        ys.append(y.numpy())
        idxs.append(idx.numpy())
        probs.append(p.cpu().numpy())
    y_true = np.concatenate(ys)
    idx = np.concatenate(idxs)
    probs = np.concatenate(probs)
    pred = probs.argmax(axis=1)
    return {"y_true": y_true, "indices": idx, "probs": probs, "y_pred": pred}

@torch.no_grad()
def infer_binary_refiner_on_full_split(model, split_loader):
    model.eval()
    idxs, p1s = [], []
    for x,y,idx in split_loader:
        x = x.to(DEVICE)
        p = F.softmax(model(x), dim=1)
        if CONFIG["USE_TTA_HFLIP"]:
            pf = F.softmax(model(torch.flip(x, dims=[3])), dim=1)
            p = 0.5 * (p + pf)
        idxs.append(idx.numpy())
        p1s.append(p[:,1].cpu().numpy())
    return np.concatenate(idxs), np.concatenate(p1s)

def tune_refiner_threshold(base_val_out, p1_val, y_val):
    base_pred = base_val_out["probs"].argmax(axis=1)
    base_m = compute_metrics(y_val, base_pred)
    best = {"score": -1e9, "thr": 0.5, "metrics": base_m}
    for thr in np.arange(0.20, 0.81, 0.02):
        pred = base_pred.copy()
        mask = np.isin(base_pred, [0,1]) | ((base_val_out["probs"][:,0] + base_val_out["probs"][:,1]) > 0.55)
        pred[mask & (p1_val >= thr)] = 1
        pred[mask & (p1_val < thr)] = 0
        m = compute_metrics(y_val, pred)
        # Keep accuracy close while boosting Grade-1 recall
        penalty = max(0.0, base_m["acc"] - 0.015 - m["acc"]) * 5.0
        score = 0.55*m["acc"] + 0.15*m["qwk"] + 0.30*m["g1_recall"] - penalty
        if score > best["score"]:
            best = {"score": score, "thr": float(thr), "metrics": m}
    print("Best refiner threshold:", best["thr"])
    print_metrics(base_m, "Base validation before refiner")
    print_metrics(best["metrics"], "Validation after refiner")
    return best

if CONFIG["RUN_GRADE01_REFINER"]:
    # Train one refiner per seed and apply it to corresponding base seed.
    refiner_rows = []
    for r in main_results:
        seed = r["summary"]["seed"]
        refiner = train_refiner(seed=seed)

        # Need full val/test loaders with same ordering as base outputs.
        _, _, _, _, val_loader_full, test_loader_full = build_loaders(CONFIG)
        val_idx, val_p1 = infer_binary_refiner_on_full_split(refiner, val_loader_full)
        test_idx, test_p1 = infer_binary_refiner_on_full_split(refiner, test_loader_full)

        # Sort p1 arrays by idx to match base output order.
        val_map = dict(zip(val_idx, val_p1))
        test_map = dict(zip(test_idx, test_p1))
        val_p1_aligned = np.array([val_map[i] for i in r["val"]["indices"]])
        test_p1_aligned = np.array([test_map[i] for i in r["test"]["indices"]])

        best = tune_refiner_threshold(r["val"], val_p1_aligned, r["val"]["y_true"])
        thr = best["thr"]

        base_test_pred = r["test"]["probs"].argmax(axis=1)
        pred = base_test_pred.copy()
        mask = np.isin(base_test_pred, [0,1]) | ((r["test"]["probs"][:,0] + r["test"]["probs"][:,1]) > 0.55)
        pred[mask & (test_p1_aligned >= thr)] = 1
        pred[mask & (test_p1_aligned < thr)] = 0
        m = compute_metrics(r["test"]["y_true"], pred)
        print_metrics(m, f"TEST after Grade-0/1 refiner seed={seed}")

        out_dir = r["out_dir"]
        pd.DataFrame({
            "idx": r["test"]["indices"],
            "y_true": r["test"]["y_true"],
            "base_pred": base_test_pred,
            "refined_pred": pred,
            "refiner_p_grade1": test_p1_aligned,
        }).to_csv(out_dir / "test_predictions_grade01_refined.csv", index=False)
        per_class_table(r["test"]["y_true"], pred).to_csv(out_dir / "test_per_class_grade01_refined.csv", index=False)

        refiner_rows.append({
            "seed": seed,
            "threshold": thr,
            **{f"test_{k}": v for k,v in m.items()},
        })

    refiner_df = pd.DataFrame(refiner_rows)
    refiner_summary_dir = OUT_DIR / f"main_{CONFIG['BACKBONE']}_{CONFIG['IMG_SIZE']}"
    refiner_summary_dir.mkdir(parents=True, exist_ok=True)
    refiner_df.to_csv(refiner_summary_dir / "summary_grade01_refiner.csv", index=False)
    display(refiner_df)

In [ ]:
# =========================
# 14) REVIEWER ABLATION SUITE — import-aware
# =========================
# Reviewer-1: SE block contribution.
# Reviewer-2: Weighted CE / Focal / balancing strategies.
#
# Important:
# - The finished 3-seed main CE/OA rows are imported from v5 when available.
# - Only missing diagnostic rows are trained here by default.
# - This avoids rerunning the expensive main CE/OA 3-seed experiments.

def arm_name(prefix, cfg):
    se = "SE" if cfg.get("USE_EXTRA_SE", False) else "noSE"
    loss = cfg.get("LOSS_MODE", "ce")
    ow = cfg.get("ORDINAL_WEIGHT", 0)
    rw = cfg.get("RANK_WEIGHT", 0)
    sampler = "sampler" if cfg.get("USE_WEIGHTED_SAMPLER", False) else "nosampler"
    return f"{prefix}_{CONFIG['BACKBONE']}_{CONFIG['IMG_SIZE']}_{se}_{loss}_{sampler}_ord{ow}_rank{rw}".replace(".", "p")

def load_existing_experiment_summary(exp_name, label, source_note="resumed_existing_output"):
    """Load a finished per-experiment summary_all_seeds.csv if it already exists in OUT_DIR."""
    summary_file = OUT_DIR / exp_name / "summary_all_seeds.csv"
    if not summary_file.exists():
        return pd.DataFrame()
    try:
        df = pd.read_csv(summary_file)
        df.insert(0, "source", source_note)
        df.insert(0, "ablation", label)
        return df
    except Exception as e:
        print(f"WARNING: could not load existing summary for {exp_name}: {repr(e)}")
        return pd.DataFrame()

def load_imported_summary_rows(exp_name, ablation_label, source_note):
    summary_file = OUT_DIR / exp_name / "summary_all_seeds.csv"
    if not summary_file.exists():
        return pd.DataFrame()
    df = pd.read_csv(summary_file)
    df.insert(0, "source", source_note)
    df.insert(0, "ablation", ablation_label)
    return df

# Imported rows from the completed v5 3-seed main run.
# They are the cleanest evidence for the two most important ablation anchors:
#   1) strong SE + weighted CE baseline
#   2) main moderate ordinal model with active CORAL + pairwise ranking
IMPORTED_ABLATION_ROWS = [
    {
        "exp_name": f"ce_baseline_{CONFIG['BACKBONE']}_{CONFIG['IMG_SIZE']}",
        "label": "SE_weightedCE_sampler_imported_3seed",
        "source": "imported_main_3seed",
    },
    {
        "exp_name": f"moderate_ordinal_{CONFIG['BACKBONE']}_{CONFIG['IMG_SIZE']}",
        "label": "SE_CORAL_pairwise_main_imported_3seed",
        "source": "imported_main_3seed",
    },
]

# Missing diagnostic rows to train now.
# These close the reviewer objections without repeating the completed main run.
ABLATION_CONFIGS_TO_RUN = [
    {
        "label": "noSE_weightedCE_sampler",
        "USE_EXTRA_SE": False,
        "LOSS_MODE": "weighted_ce",
        "ORDINAL_WEIGHT": 0.0,
        "RANK_WEIGHT": 0.0,
        "ORDINAL_PROB_BLEND": 0.0,
        "USE_WEIGHTED_SAMPLER": True,
        "SAMPLER_POWER": 0.25,
    },
    {
        "label": "SE_CE_no_sampler",
        "USE_EXTRA_SE": True,
        "LOSS_MODE": "ce",
        "ORDINAL_WEIGHT": 0.0,
        "RANK_WEIGHT": 0.0,
        "ORDINAL_PROB_BLEND": 0.0,
        "USE_WEIGHTED_SAMPLER": False,
    },
    {
        "label": "SE_focal_sampler",
        "USE_EXTRA_SE": True,
        "LOSS_MODE": "focal",
        "ORDINAL_WEIGHT": 0.0,
        "RANK_WEIGHT": 0.0,
        "ORDINAL_PROB_BLEND": 0.0,
        "USE_WEIGHTED_SAMPLER": True,
        "SAMPLER_POWER": 0.25,
    },
    {
        "label": "SE_CORAL_only",
        "USE_EXTRA_SE": True,
        "LOSS_MODE": "weighted_ce",
        "ORDINAL_WEIGHT": 0.15,
        "RANK_WEIGHT": 0.0,
        "ORDINAL_PROB_BLEND": 0.10,
        "USE_WEIGHTED_SAMPLER": True,
        "SAMPLER_POWER": 0.25,
    },
    {
        "label": "SE_CORAL_pairwise_strong",
        "USE_EXTRA_SE": True,
        "LOSS_MODE": "weighted_ce",
        "ORDINAL_WEIGHT": 0.35,
        "RANK_WEIGHT": 0.20,
        "ORDINAL_PROB_BLEND": 0.25,
        "PAIRWISE_MARGIN": 1.0,
        "USE_WEIGHTED_SAMPLER": True,
        "SAMPLER_POWER": 0.25,
    },
]

def run_ablation_suite():
    rows = []

    if CONFIG.get("USE_IMPORTED_MAIN_FOR_REVIEWER_TABLES", True):
        for item in IMPORTED_ABLATION_ROWS:
            imported = load_imported_summary_rows(item["exp_name"], item["label"], item["source"])
            if len(imported):
                rows.append(imported)
                print(f"Imported ablation anchor: {item['label']} from {item['exp_name']}")
            else:
                print(f"WARNING: imported ablation anchor not found: {item['exp_name']}")

    for item in ABLATION_CONFIGS_TO_RUN:
        cfg = copy.deepcopy(CONFIG)
        cfg.update({k:v for k,v in item.items() if k != "label"})
        cfg["SEEDS"] = CONFIG["REVIEWER_TABLE_SEEDS"]
        cfg["EPOCHS"] = CONFIG["REVIEWER_TABLE_EPOCHS"]
        cfg["PATIENCE"] = CONFIG["REVIEWER_TABLE_PATIENCE"]

        exp_name = arm_name("ablation_" + item["label"], cfg)

        if CONFIG.get("RESUME_COMPLETED_REVIEWER_RUNS", True):
            existing = load_existing_experiment_summary(exp_name, item["label"])
            if len(existing):
                rows.append(existing)
                print(f"Resumed existing ablation output without retraining: {item['label']} from {exp_name}")
                continue

        for seed in cfg["SEEDS"]:
            res = train_one_seed(cfg, seed=seed, experiment_name=exp_name)
            rows.append(pd.DataFrame([{
                "ablation": item["label"],
                "source": "trained_reviewer_table",
                **res["summary"],
            }]))

    if not rows:
        print("No ablation rows were produced.")
        return pd.DataFrame(), pd.DataFrame()

    df = pd.concat(rows, ignore_index=True, sort=False)
    df.to_csv(OUT_DIR / "ablation_summary_all_rows.csv", index=False)

    metric_cols = [c for c in df.columns if c.startswith("val_") or c.startswith("test_")]
    keep_cols = ["ablation", "source", "seed"] + metric_cols
    df_compact = df[[c for c in keep_cols if c in df.columns]].copy()
    df_compact.to_csv(OUT_DIR / "ablation_summary_compact.csv", index=False)

    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    # Keep only numeric metric/config columns for mean/std; object columns such as "source" break pandas aggregation.
    summary = df.groupby("ablation")[numeric_cols].agg(["mean", "std"])
    summary.to_csv(OUT_DIR / "ablation_summary_mean_std.csv")
    display(df_compact)
    display(summary)
    return df, summary

if CONFIG["RUN_FULL_ABLATION"]:
    ablation_df, ablation_summary = run_ablation_suite()
else:
    print("Ablation suite is OFF. Set CONFIG['RUN_FULL_ABLATION']=True to run reviewer ablations.")


In [ ]:
# =========================
# 15) LAMBDA / DELTA SENSITIVITY — import-aware
# =========================
# Reviewer-2 asked for sensitivity of λ and δ.
#
# The main λ=0.15 / rank=0.08 / δ=1.0 setting is imported from the completed 3-seed run.
# Additional settings are trained as reviewer-table diagnostics.

def load_existing_sensitivity_summary(exp_name, grid_item, sensitivity_id, source_note="resumed_existing_output"):
    """Load a finished sensitivity run if summary_all_seeds.csv already exists in OUT_DIR."""
    summary_file = OUT_DIR / exp_name / "summary_all_seeds.csv"
    if not summary_file.exists():
        return pd.DataFrame()
    try:
        df = pd.read_csv(summary_file)
        df.insert(0, "source", source_note)
        df.insert(0, "ordinal_prob_blend", grid_item.get("ORDINAL_PROB_BLEND", np.nan))
        df.insert(0, "delta", grid_item.get("PAIRWISE_MARGIN", np.nan))
        df.insert(0, "lambda_rank", grid_item.get("RANK_WEIGHT", np.nan))
        df.insert(0, "lambda_coral", grid_item.get("ORDINAL_WEIGHT", np.nan))
        df.insert(0, "sensitivity_id", sensitivity_id)
        return df
    except Exception as e:
        print(f"WARNING: could not load existing sensitivity summary for {exp_name}: {repr(e)}")
        return pd.DataFrame()

def load_imported_main_sensitivity_row():
    exp_name = f"moderate_ordinal_{CONFIG['BACKBONE']}_{CONFIG['IMG_SIZE']}"
    summary_file = OUT_DIR / exp_name / "summary_all_seeds.csv"
    if not summary_file.exists():
        return pd.DataFrame()
    df = pd.read_csv(summary_file)
    df.insert(0, "source", "imported_main_3seed")
    df.insert(0, "ordinal_prob_blend", 0.10)
    df.insert(0, "delta", 1.0)
    df.insert(0, "lambda_rank", 0.08)
    df.insert(0, "lambda_coral", 0.15)
    df.insert(0, "sensitivity_id", "main_imported")
    return df

SENSITIVITY_GRID_TO_RUN = [
    {"ORDINAL_WEIGHT": 0.05, "RANK_WEIGHT": 0.02, "PAIRWISE_MARGIN": 0.5, "ORDINAL_PROB_BLEND": 0.05},
    {"ORDINAL_WEIGHT": 0.10, "RANK_WEIGHT": 0.05, "PAIRWISE_MARGIN": 1.0, "ORDINAL_PROB_BLEND": 0.08},
    {"ORDINAL_WEIGHT": 0.25, "RANK_WEIGHT": 0.10, "PAIRWISE_MARGIN": 1.0, "ORDINAL_PROB_BLEND": 0.15},
    {"ORDINAL_WEIGHT": 0.35, "RANK_WEIGHT": 0.20, "PAIRWISE_MARGIN": 1.0, "ORDINAL_PROB_BLEND": 0.25},
    {"ORDINAL_WEIGHT": 0.35, "RANK_WEIGHT": 0.20, "PAIRWISE_MARGIN": 1.5, "ORDINAL_PROB_BLEND": 0.25},
]

def run_sensitivity_suite():
    rows = []

    if CONFIG.get("USE_IMPORTED_MAIN_FOR_REVIEWER_TABLES", True):
        imported = load_imported_main_sensitivity_row()
        if len(imported):
            rows.append(imported)
            print("Imported main sensitivity anchor: lambda=0.15, rank=0.08, delta=1.0")
        else:
            print("WARNING: imported main sensitivity anchor not found.")

    for i, grid_item in enumerate(SENSITIVITY_GRID_TO_RUN, start=1):
        cfg = copy.deepcopy(CONFIG)
        cfg.update(grid_item)
        cfg["SEEDS"] = CONFIG["REVIEWER_TABLE_SEEDS"]
        cfg["EPOCHS"] = CONFIG["REVIEWER_TABLE_EPOCHS"]
        cfg["PATIENCE"] = CONFIG["REVIEWER_TABLE_PATIENCE"]
        cfg["USE_EXTRA_SE"] = True
        cfg["LOSS_MODE"] = "weighted_ce"
        cfg["USE_WEIGHTED_SAMPLER"] = True
        cfg["SAMPLER_POWER"] = 0.25

        name = (
            f"sens_{CONFIG['BACKBONE']}_{CONFIG['IMG_SIZE']}"
            f"_ow{cfg['ORDINAL_WEIGHT']}_rw{cfg['RANK_WEIGHT']}_d{cfg['PAIRWISE_MARGIN']}"
        ).replace(".", "p")

        if CONFIG.get("RESUME_COMPLETED_REVIEWER_RUNS", True):
            existing = load_existing_sensitivity_summary(name, grid_item, i)
            if len(existing):
                rows.append(existing)
                print(f"Resumed existing sensitivity output without retraining: {name}")
                continue

        for seed in cfg["SEEDS"]:
            res = train_one_seed(cfg, seed=seed, experiment_name=name)
            rows.append(pd.DataFrame([{
                "sensitivity_id": i,
                "source": "trained_reviewer_table",
                "lambda_coral": cfg["ORDINAL_WEIGHT"],
                "lambda_rank": cfg["RANK_WEIGHT"],
                "delta": cfg["PAIRWISE_MARGIN"],
                "ordinal_prob_blend": cfg["ORDINAL_PROB_BLEND"],
                **res["summary"],
            }]))

    if not rows:
        print("No sensitivity rows were produced.")
        return pd.DataFrame(), pd.DataFrame()

    df = pd.concat(rows, ignore_index=True, sort=False)
    df.to_csv(OUT_DIR / "lambda_delta_sensitivity_all_rows.csv", index=False)

    metric_cols = [c for c in df.columns if c.startswith("val_") or c.startswith("test_")]
    keep_cols = ["sensitivity_id", "source", "lambda_coral", "lambda_rank", "delta", "ordinal_prob_blend", "seed"] + metric_cols
    df_compact = df[[c for c in keep_cols if c in df.columns]].copy()
    df_compact.to_csv(OUT_DIR / "lambda_delta_sensitivity_compact.csv", index=False)

    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    group_cols = ["lambda_coral", "lambda_rank", "delta", "ordinal_prob_blend"]
    value_cols = [c for c in numeric_cols if c not in group_cols]
    summary = df.groupby(group_cols)[value_cols].agg(["mean", "std"])
    summary.to_csv(OUT_DIR / "lambda_delta_sensitivity_mean_std.csv")
    display(df_compact)
    display(summary)
    return df, summary

if CONFIG["RUN_LAMBDA_DELTA_SENSITIVITY"]:
    sens_df, sens_summary = run_sensitivity_suite()
else:
    print("Lambda/delta sensitivity is OFF. Set CONFIG['RUN_LAMBDA_DELTA_SENSITIVITY']=True to run it.")


In [ ]:
# =========================
# 16) LIGHTWEIGHT / COMPLEXITY MEASUREMENT
# =========================

def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

@torch.no_grad()
def measure_inference_time(model, img_size=224, batch_size=32, warmup=10, iters=30):
    model.eval()
    x = torch.randn(batch_size, 3, img_size, img_size, device=DEVICE)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    for _ in range(warmup):
        _ = model(x)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(iters):
        _ = model(x)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    elapsed = time.time() - t0
    ms_per_image = elapsed / (iters * batch_size) * 1000
    return ms_per_image

def try_flops(model, img_size=224):
    try:
        from thop import profile
        dummy = torch.randn(1, 3, img_size, img_size).to(DEVICE)
        flops, params = profile(model, inputs=(dummy,), verbose=False)
        return float(flops), float(params)
    except Exception as e:
        return None, None

def complexity_report(configs=None):
    if configs is None:
        configs = [
            {"name": "EffNet-B0 no extra SE", "BACKBONE": "efficientnet_b0", "USE_EXTRA_SE": False},
            {"name": "EffNet-B0 + extra SE", "BACKBONE": "efficientnet_b0", "USE_EXTRA_SE": True},
            {"name": "EffNet-B3 + extra SE", "BACKBONE": "efficientnet_b3", "USE_EXTRA_SE": True},
            {"name": "EffNet-B4 + extra SE", "BACKBONE": "efficientnet_b4", "USE_EXTRA_SE": True},
            {"name": "ResNet50 + extra SE", "BACKBONE": "resnet50", "USE_EXTRA_SE": True},
        ]
    rows = []
    for c in configs:
        model = OAHybridNet(
            backbone_name=c["BACKBONE"],
            pretrained=False,  # complexity does not need pretrained download
            use_extra_se=c["USE_EXTRA_SE"],
        ).to(DEVICE)
        total, trainable = count_params(model)
        ms = measure_inference_time(model, img_size=CONFIG["IMG_SIZE"], batch_size=min(CONFIG["BATCH_SIZE"], 32))
        flops, _ = try_flops(model, img_size=CONFIG["IMG_SIZE"])
        rows.append({
            "model": c["name"],
            "params_total_M": total / 1e6,
            "params_trainable_M": trainable / 1e6,
            "inference_ms_per_image": ms,
            "flops_G": None if flops is None else flops / 1e9,
            "note": "FLOPs requires thop; blank if thop is unavailable.",
        })
        del model
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
    df = pd.DataFrame(rows)
    df.to_csv(OUT_DIR / "complexity_report.csv", index=False)
    return df

complexity_df = complexity_report()
display(complexity_df)

In [ ]:
# =========================
# 17) BOOTSTRAP CI FOR FINAL ENSEMBLE OUTPUTS
# =========================
# Computes per-metric bootstrap CIs for every test ensemble prediction file generated above.

def load_prediction_csv(path):
    df = pd.read_csv(path)
    pred_col = None
    for c in ["y_pred", "y_pred_after_bias", "refined_pred"]:
        if c in df.columns:
            pred_col = c
    if pred_col is None:
        raise ValueError(f"No prediction column found in {path}.")
    return df["y_true"].values, df[pred_col].values

ci_rows = []
candidate_files = sorted(OUT_DIR.rglob("test_ensemble*_predictions.csv"))

if candidate_files:
    for f in candidate_files:
        y_true, y_pred = load_prediction_csv(f)
        print("Bootstrap CI for:", f)
        ci_df = bootstrap_ci(y_true, y_pred, n_boot=1000, seed=42)
        ci_df.insert(0, "file", str(f))
        ci_df.to_csv(f.with_name(f.stem + "_bootstrap_ci.csv"), index=False)
        ci_rows.append(ci_df)

        pct = per_class_table(y_true, y_pred)
        pct.to_csv(f.with_name(f.stem + "_per_class_recomputed.csv"), index=False)

    all_ci_df = pd.concat(ci_rows, ignore_index=True)
    all_ci_df.to_csv(OUT_DIR / "bootstrap_ci_all_ensemble_outputs.csv", index=False)
    display(all_ci_df)
else:
    print("No ensemble prediction file found yet.")


In [ ]:
# =========================
# 18) OPTIONAL GRAD-CAM FOR GRADE-1 FAILURES
# =========================
# This is intentionally lightweight. It creates Grad-CAM-like maps from the last convolutional layer.
# Use after training. It is meant for:
# - correct Grade-1 examples
# - Grade-1 misclassified as Grade-0
# - Grade-0 misclassified as Grade-1

import matplotlib.pyplot as plt

def find_last_conv_module(model):
    last_name, last_module = None, None
    for name, module in model.named_modules():
        if isinstance(module, nn.Conv2d):
            last_name, last_module = name, module
    return last_name, last_module

class GradCAM:
    def __init__(self, model):
        self.model = model
        self.model.eval()
        self.activations = None
        self.gradients = None
        name, module = find_last_conv_module(model)
        if module is None:
            raise RuntimeError("No Conv2d layer found for Grad-CAM.")
        self.layer_name = name
        self.h1 = module.register_forward_hook(self.forward_hook)
        self.h2 = module.register_full_backward_hook(self.backward_hook)

    def forward_hook(self, module, inp, out):
        self.activations = out.detach()

    def backward_hook(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def remove(self):
        self.h1.remove()
        self.h2.remove()

    def __call__(self, x, target_class=None):
        self.model.zero_grad(set_to_none=True)
        outputs = self.model(x)
        logits = outputs["cls_logits"]
        if target_class is None:
            target_class = int(logits.argmax(dim=1).item())
        score = logits[:, target_class].sum()
        score.backward()

        weights = self.gradients.mean(dim=(2,3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)
        cam = cam[0,0].detach().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, target_class

def denormalize_tensor(x):
    mean = torch.tensor(IMAGENET_MEAN).view(3,1,1)
    std = torch.tensor(IMAGENET_STD).view(3,1,1)
    img = x.cpu() * std + mean
    img = img.clamp(0,1).permute(1,2,0).numpy()
    return img

def plot_gradcam_examples(model, dataset, pred_df, save_dir, max_per_group=4):
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    groups = {
        "grade1_correct": pred_df[(pred_df.y_true == 1) & (pred_df.y_pred == 1)].head(max_per_group),
        "grade1_as_grade0": pred_df[(pred_df.y_true == 1) & (pred_df.y_pred == 0)].head(max_per_group),
        "grade0_as_grade1": pred_df[(pred_df.y_true == 0) & (pred_df.y_pred == 1)].head(max_per_group),
    }

    cammer = GradCAM(model)
    for group_name, rows in groups.items():
        for _, row in rows.iterrows():
            idx = int(row["idx"]) if "idx" in row else int(row.name)
            x, y, _ = dataset[idx]
            x1 = x.unsqueeze(0).to(DEVICE)
            cam, target = cammer(x1, target_class=int(row["y_pred"]))
            img = denormalize_tensor(x)

            plt.figure(figsize=(5,5))
            plt.imshow(img, cmap="gray")
            plt.imshow(cam, alpha=0.35)
            plt.title(f"{group_name}: true={int(row.y_true)}, pred={int(row.y_pred)}")
            plt.axis("off")
            out_path = save_dir / f"{group_name}_idx{idx}_true{int(row.y_true)}_pred{int(row.y_pred)}.png"
            plt.savefig(out_path, dpi=220, bbox_inches="tight")
            plt.close()
    cammer.remove()
    print("Saved Grad-CAM images to:", save_dir)

# Example usage after main training:
# best_model = ordinal_model_results[0]["model"]
# pred_file = OUT_DIR / MAIN_EXPERIMENTS[1]["name"] / "test_ensemble_bias_predictions.csv"
# pred_df = pd.read_csv(pred_file)
# _, _, test_ds_eval, _, _, _ = build_loaders(CONFIG)
# plot_gradcam_examples(best_model, test_ds_eval, pred_df, OUT_DIR / "gradcam_grade1_failures")

In [ ]:
# =========================
# 19) FINAL REPORT HELPER
# =========================

def collect_all_summaries(out_dir=OUT_DIR):
    rows = []
    for f in Path(out_dir).rglob("summary*.csv"):
        try:
            df = pd.read_csv(f)
            df.insert(0, "file", str(f))
            rows.append(df)
        except Exception:
            pass
    if not rows:
        return pd.DataFrame()
    return pd.concat(rows, ignore_index=True, sort=False)

all_summaries = collect_all_summaries()
if len(all_summaries):
    all_summaries.to_csv(OUT_DIR / "ALL_SUMMARIES_COLLECTED.csv", index=False)
    display(all_summaries)
else:
    print("No summary files found yet.")


## Anonymous repo package export

Bu bölüm notebook sonunda çalışır ve makale dosyalarını içermeyen anonim repo paketi üretir.

Paket içeriği:

- `configs/`: final config JSON
- `notebooks/`: bu notebookun kopyasını manuel koymak için README
- `src/`: minimal metric/reproducibility yardımcıları
- `outputs/metrics/`: summary/per-class/confusion/bootstrap/history CSV/JSON
- `outputs/predictions/`: validation/test prediction CSV dosyaları
- `outputs/figures/`: PNG/JPG/SVG figürleri
- `outputs/manifests/`: run ve output manifest JSON dosyaları
- `docs/`: anonim release kontrol listesi

Makale PDF/DOCX/LaTeX, cover letter ve reviewer response dosyaları bilinçli olarak dahil edilmez.


In [ ]:
# =========================
# 20) CREATE ANONYMOUS REPOSITORY PACKAGE
# =========================

import platform
from datetime import datetime
import shutil
import zipfile

def safe_copy(src, dst):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        stem, suffix = dst.stem, dst.suffix
        dst = dst.with_name(f"{stem}_{abs(hash(str(src))) % 100000}{suffix}")
    shutil.copy2(src, dst)
    return dst

def should_exclude_from_repo(path: Path, repo_root: Path):
    lower = path.name.lower()
    suffix = path.suffix.lower()

    # Avoid recursive copying of the repo package into itself.
    try:
        path.relative_to(repo_root)
        return True
    except Exception:
        pass

    # Exclude heavy models/checkpoints and manuscript/submission files.
    if suffix in {".pt", ".pth", ".ckpt", ".onnx", ".doc", ".docx", ".tex", ".bib", ".pdf"}:
        return True
    blocked_tokens = [
        "manuscript", "marked", "clear", "cover", "response_to_reviewers",
        "reviewer", "editor", "submission", "redacted-token", "latex"
    ]
    if any(tok in lower for tok in blocked_tokens):
        return True
    return False

def create_minimal_repo_files(repo_root: Path):
    repo_root.mkdir(parents=True, exist_ok=True)
    for sub in [
        "configs", "notebooks", "src/ordinal_component_model", "scripts",
        "outputs/metrics", "outputs/figures", "outputs/predictions",
        "outputs/logs", "outputs/manifests", "docs"
    ]:
        (repo_root / sub).mkdir(parents=True, exist_ok=True)

    (repo_root / "README.md").write_text("""# Anonymous Reproducibility Repository

This anonymous repository contains the reproducibility materials for knee osteoarthritis severity grading experiments, including the main CE/OA comparison and reviewer-closure ablation/sensitivity tables.

It intentionally includes code, configurations, metrics, predictions, figures and manifests only. It does not include manuscript PDFs, LaTeX sources, Word files, cover letters, response letters, or journal correspondence.

## Expected dataset layout

```text
DATA_ROOT/
├── train/0 ... train/4
├── val/0   ... val/4
└── test/0  ... test/4
```

The public dataset should be obtained separately and is not redistributed here.

## Main experiment and reviewer closure

The main final run compares two arms under identical backbone, image size, seeds, preprocessing, and schedule. The reviewer-closure run adds ablation, sensitivity, complexity, bootstrap, and manifest outputs:

- CE baseline: EfficientNet-B4, 380 px, SE, weighted CE
- moderate ordinal model: same setup plus active CORAL and pairwise ranking losses

The package also includes validation-based class-bias tuning, probability averaging across seeds, per-class results, bootstrap CIs, and significance-test summaries.

## Output locations

- `outputs/metrics/`: summary, per-class, confusion matrix, bootstrap, and history tables
- `outputs/predictions/`: validation/test predictions
- `outputs/figures/`: generated figures
- `outputs/manifests/`: run and output manifests
""", encoding="utf-8")

    (repo_root / "requirements.txt").write_text("""torch
torchvision
numpy
pandas
scikit-learn
matplotlib
Pillow
tqdm
opencv-python
jupyter
papermill
""", encoding="utf-8")

    (repo_root / ".gitignore").write_text("""__pycache__/
*.py[cod]
.ipynb_checkpoints/
*.pt
*.pth
*.ckpt
*.onnx
data/
datasets/
train/
val/
test/
*.doc
*.docx
*.pdf
*.tex
*.bib
cover_letter*
response_to_reviewers*
manuscript*
marked*
clear*
.DS_Store
Thumbs.db
""", encoding="utf-8")

    (repo_root / "LICENSE").write_text("""MIT License

Copyright (c) 2026 Anonymous

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files, to deal in the Software
without restriction, subject to the conditions of the MIT License.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND.
""", encoding="utf-8")

    (repo_root / "src/ordinal_component_model/__init__.py").write_text('__version__ = "0.1.0"\n', encoding="utf-8")
    (repo_root / "src/ordinal_component_model/metrics.py").write_text("""import numpy as np
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, mean_absolute_error, cohen_kappa_score, precision_recall_fscore_support

def compute_basic_metrics(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=list(range(5)), zero_division=0
    )
    return {
        "acc": float(accuracy_score(y_true, y_pred)),
        "qwk": float(cohen_kappa_score(y_true, y_pred, weights="quadratic")),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "bacc": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "g1_recall": float(recall[1]),
        "g1_precision": float(precision[1]),
        "g1_f1": float(f1[1]),
    }
""", encoding="utf-8")

    (repo_root / "docs/anonymous_release_checklist.md").write_text("""# Anonymous Release Checklist

- [ ] No author-identifying manuscript files are included.
- [ ] No PDF/DOCX/LaTeX/cover-letter/response-letter files are included.
- [ ] No raw dataset files are redistributed.
- [ ] No checkpoints are included by default.
- [ ] Config, metrics, predictions, figures, and manifests are included.
- [ ] Paths do not reveal local usernames or author identity.
""", encoding="utf-8")

def _redact_for_repo(obj):
    # Remove user-specific Kaggle input paths from public config/manifests.
    if isinstance(obj, dict):
        out = {}
        for k, v in obj.items():
            if k in {"PREVIOUS_OUTPUTS_ROOT"}:
                out[k] = "<KAGGLE_INPUT_PREVIOUS_OUTPUTS>"
            elif k == "DATA_ROOT" and isinstance(v, str) and v:
                out[k] = "<DATA_ROOT>"
            else:
                out[k] = _redact_for_repo(v)
        return out
    if isinstance(obj, list):
        return [_redact_for_repo(v) for v in obj]
    if isinstance(obj, str):
        # Keep common public dataset hints out of the anonymous package too.
        if "/kaggle/input" in obj:
            return "<KAGGLE_INPUT_PATH>"
    return obj

def write_config_and_run_manifest(repo_root: Path):
    cfg_name = f"reviewer_closure_{CONFIG['BACKBONE']}_{CONFIG['IMG_SIZE']}.json"
    config_to_save = _redact_for_repo(dict(CONFIG))
    config_to_save["DATA_ROOT_RESOLVED"] = "<DATA_ROOT>"
    config_to_save["SPLITS_RESOLVED"] = {k: f"<DATA_ROOT>/{k}" for k in SPLITS.keys()}

    (repo_root / "configs" / cfg_name).write_text(json.dumps(config_to_save, indent=2), encoding="utf-8")
    (repo_root / "configs" / "main_experiment_arms.json").write_text(json.dumps(_redact_for_repo(MAIN_EXPERIMENTS), indent=2), encoding="utf-8")

    # Copy the executed notebook into the anonymous repo if Kaggle exposes it.
    for nb_candidate in [Path("__notebook__.ipynb"), Path("/kaggle/working/__notebook__.ipynb")]:
        if nb_candidate.exists():
            safe_copy(nb_candidate, repo_root / "notebooks" / "ablation_sensitivity_executed.ipynb")
            break

    manifest = {
        "created_at": datetime.utcnow().isoformat() + "Z",
        "python": platform.python_version(),
        "platform": platform.platform(),
        "torch": torch.__version__,
        "device": str(DEVICE),
        "cuda_available": bool(torch.cuda.is_available()),
        "cuda_device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "data_root": "<DATA_ROOT>",
        "splits": {k: f"<DATA_ROOT>/{k}" for k in SPLITS.keys()},
        "config_file": f"configs/{cfg_name}",
        "main_experiments": _redact_for_repo(MAIN_EXPERIMENTS),
        "anonymous_note": "This package excludes manuscript/submission files and redacts local/Kaggle input paths.",
    }
    (repo_root / "outputs/manifests/run_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

def copy_outputs_to_repo(out_dir: Path, repo_root: Path):
    copied = []
    patterns_by_category = {
        "metrics": ["*summary*.csv", "*per_class*.csv", "*confusion_matrix*.csv", "*bootstrap*.csv", "*diffs*.csv", "split_distribution.csv", "*history*.csv", "*.json"],
        "predictions": ["*predictions*.csv", "*pred*.csv"],
        "figures": ["*.png", "*.jpg", "*.jpeg", "*.svg"],
        "logs": ["*.log", "*history*.csv"],
    }

    for category, patterns in patterns_by_category.items():
        dst_base = repo_root / "outputs" / category
        seen = set()
        for pattern in patterns:
            for src in out_dir.rglob(pattern):
                if not src.is_file():
                    continue
                if should_exclude_from_repo(src, repo_root):
                    continue
                # Do not copy files from package zip if rerun.
                if CONFIG["REPO_PACKAGE_NAME"] in src.parts:
                    continue
                key = str(src.resolve())
                if key in seen:
                    continue
                seen.add(key)
                dst = safe_copy(src, dst_base / src.name)
                copied.append({
                    "category": category,
                    "source": str(src),
                    "destination": str(dst.relative_to(repo_root)),
                    "size_bytes": dst.stat().st_size,
                })

    output_manifest = {
        "created_at": datetime.utcnow().isoformat() + "Z",
        "num_files_copied": len(copied),
        "files": copied,
    }
    (repo_root / "outputs/manifests/output_manifest.json").write_text(json.dumps(output_manifest, indent=2), encoding="utf-8")
    return output_manifest

def zip_repo_package(repo_root: Path):
    zip_path = repo_root.with_suffix(".zip")
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for path in repo_root.rglob("*"):
            if not path.is_file():
                continue
            if should_exclude_from_repo(path, repo_root):
                continue
            zf.write(path, path.relative_to(repo_root.parent))
    return zip_path

if CONFIG.get("MAKE_ANON_REPO_PACKAGE", True):
    repo_root = OUT_DIR / CONFIG["REPO_PACKAGE_NAME"]
    if repo_root.exists():
        shutil.rmtree(repo_root)

    create_minimal_repo_files(repo_root)
    write_config_and_run_manifest(repo_root)
    output_manifest = copy_outputs_to_repo(OUT_DIR, repo_root)
    zip_path = zip_repo_package(repo_root)

    print("Anonymous repo package created:")
    print("  folder:", repo_root.resolve())
    print("  zip   :", zip_path.resolve())
    print("  copied files:", output_manifest["num_files_copied"])
else:
    print("CONFIG['MAKE_ANON_REPO_PACKAGE'] is False; repo package was not created.")


## Makalede nasıl anlatılmalı? — v6

Bu notebook iki şeyi birleştirir:

1. Önceki tamamlanmış **3-seed ana CE vs moderate ordinal model** sonuçları.
2. Yeni çalıştırılan **reviewer-closure ablation ve λ/δ sensitivity** tabloları.

Makaledeki güvenli anlatım:

> The final experiments compare a strong EfficientNet-B4 baseline and the proposed ordinally regularized variant under the same backbone, resolution, preprocessing, and seeds. Additional reviewer-closure analyses evaluate the effect of SE attention, imbalance-aware losses, CORAL regularization, pairwise ranking, and λ/δ sensitivity.

B4 modeli için **lightweight** demeyin. Bunun yerine:

> stronger single-model revision

veya

> higher-capacity single-model configuration

deyin.

Ablation ve sensitivity tek seed kalırsa bunu dürüstçe yazın:

> Main CE/OA comparisons were averaged over three seeds, whereas extended ablation and sensitivity analyses were run as controlled diagnostic experiments due to the computational cost of B4/380 training.

Reviewer tabloları 3 seed yapılırsa bu cümleye gerek kalmaz.
